# MEFF — measured latency of all four variants

Measures parameters, FLOPs and **latency** (batch 1) of DSG, DSG+SE, DSG+HC and the MEFF expert, each as **one network** and as a **three-network hard-vote ensemble**, in fp32 and in mixed precision (AMP, as used for the test predictions).

* **Network only**: forward passes on a random 224 × 224 input (100 timed runs after 20 warm-up runs).
* **End to end**: 200 test images, each read from disk → 42-D descriptor on the CPU (HC variants) → resize + normalise → forward pass(es) → softmax → hard vote → class on the CPU.
* Weights are random (no checkpoints needed): the computation, and therefore the latency, does not depend on the weight values.

**Settings:** Accelerator **GPU T4** (one GPU is used), Internet not needed. **Add Input:** the 5,955-image DFUC2021 dataset used for training. Then **Save Version → Save & Run All**. About 10 minutes. Download `latency_results.zip` from the Output tab.

In [ ]:
import os, hashlib, torch
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 and run again"
print("GPU:", torch.cuda.get_device_name(0))
N_IMAGES, N_WARMUP, N_RUNS = 200, 20, 100
W = "/kaggle/working/latency"; os.makedirs(W, exist_ok=True)

def find_data(root="/kaggle/input"):
    for r, d, f in os.walk(root):
        if all(c in d for c in ("NONE", "INFECTION", "ISCHAEMIA", "BOTH")):
            return r
    raise SystemExit("Image folder with NONE/INFECTION/ISCHAEMIA/BOTH not found - add the 5,955-image dataset as Input")
DATA_DIR = find_data()
n = sum(len(os.listdir(os.path.join(DATA_DIR, c))) for c in ("NONE", "INFECTION", "ISCHAEMIA", "BOTH"))
print("images in", DATA_DIR, ":", n)
assert n == 5955, "expected the 5,955-image dataset"


## 1. Model code — `meff_v3_1.py` (the exact file used for the ablation runs; do not edit)

In [ ]:
%%writefile /kaggle/working/latency/meff_v3_1.py
#!/usr/bin/env python3
"""
MEFF v3.1 pipeline = MEFF v3 (meff_v3.py, MD5 d4a5febc...) + two ablation variants + the pre-registered ablation analysis.
Nothing else is changed: the training recipe (CFG), data, split, metrics and the meff/dsg models are identical to v3.

  train       train ONE model (variant meff|dsg, one seed); resumable; stops cleanly before the wall limit
  eval        ensembles, metrics, paired group bootstrap, McNemar+Holm, decision rule (reads saved predictions)
  verify      check the dataset folder matches the frozen split (5,955 files, MD5s) before any training
  complexity  parameters, FLOPs and latency for every variant (and the 3-model MEFF ensemble)
  ablation    PROTOCOL_ABLATION.md: SE x handcrafted 2x2 ablation on the 12 runs (dsg, dsg_se, dsg_hc, meff x 3 seeds)

Variants
  meff : DenseNet-169 -> CCDGS -> Triplet Attention -> SE -> GAP -> [concat 42-D handcrafted] -> head
  dsg  : DenseNet-169 -> CCDGS -> Triplet Attention        -> GAP ->                          head
         (Dense-ShuffleGCANet re-implementation; Ajay, Bisht, Karthik, IEEE Access 13, 2025)
  dsg_se : DSG + SE                       (ablation)
  dsg_hc : DSG + 42-D handcrafted features (ablation)

Examples
  python meff_v3_1.py train --variant meff --seed 42 --data_dir DFUC2021_deduplicated_5955 \
         --split_dir MEFF_v3/split --out_dir MEFF_v3/runs/meff_s42
  python meff_v3_1.py eval --runs_dir MEFF_v3/runs --split_dir MEFF_v3/split --out_dir MEFF_v3/results
  python meff_v3_1.py complexity --out_dir MEFF_v3/results
"""
import argparse, hashlib, json, os, platform, random, sys, time

import numpy as np
import pandas as pd

CLASSES = ["NONE", "INFECTION", "ISCHAEMIA", "BOTH"]
NC = len(CLASSES)
VARIANTS = ["meff", "dsg", "dsg_se", "dsg_hc"]
USES_SE = {"meff": True, "dsg": False, "dsg_se": True, "dsg_hc": False}
USES_HC = {"meff": True, "dsg": False, "dsg_se": False, "dsg_hc": True}

# ----------------------------------------------------------------------------------------------
# Frozen training recipe (pre-registered in PROTOCOL_v3.md - do not change after Part A)
# ----------------------------------------------------------------------------------------------
CFG = dict(
    img_size=224, batch_size=32, max_epochs=100, early_stop_patience=15,
    optimizer="Adam", lr=5e-5, weight_decay=1e-3,
    scheduler="ReduceLROnPlateau(mode=max on val macro-F1, factor=0.5, patience=5, min_lr=1e-7)",
    loss="FocalLoss(gamma=2.0, no alpha)", sampler="WeightedRandomSampler (inverse class frequency)",
    dropout=0.5, hidden=512, hc_dim=42, ccdgs_groups=4, se_reduction=16, ta_kernel=7,
    train_aug="RandomResizedCrop(224,0.8-1.0), HFlip, VFlip, Rotation(15), ColorJitter(0.2,0.2,0.2,0.02)",
    eval_aug="none (ToTensor + ImageNet normalisation)",
    checkpoint_selection="best validation macro-F1",
)


def md5_file(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def log(*a):
    print(time.strftime("[%Y-%m-%d %H:%M:%S]"), *a, flush=True)


# ----------------------------------------------------------------------------------------------
# Handcrafted 42-D descriptor: LBP (10) + RGB colour histogram (24) + edge-orientation histogram (8)
# ----------------------------------------------------------------------------------------------
def handcrafted_42(rgb):
    import cv2
    from skimage.feature import local_binary_pattern
    assert rgb.dtype == np.uint8 and rgb.ndim == 3 and rgb.shape[2] == 3
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    # LBP uniform P=8,R=1 -> codes 0..9 -> 10 bins
    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    lbp_h = np.histogram(lbp, bins=np.arange(11))[0].astype(np.float64)
    lbp_h /= max(lbp_h.sum(), 1)
    # RGB colour histogram, 8 bins per channel
    col = []
    for ch in range(3):
        h = np.histogram(rgb[:, :, ch], bins=8, range=(0, 256))[0].astype(np.float64)
        col.append(h / max(h.sum(), 1))
    # Edge-orientation histogram: Sobel orientation at Canny edge pixels, 8 bins over [0, pi)
    eq = cv2.equalizeHist(cv2.GaussianBlur(gray, (5, 5), 0))  # blur + histogram equalisation (as in original MEFF)
    med = float(np.median(eq))
    edges = cv2.Canny(eq, int(max(0, 0.66 * med)), int(min(255, 1.33 * med)))
    gx = cv2.Sobel(eq, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(eq, cv2.CV_64F, 0, 1, ksize=3)
    ang = np.mod(np.arctan2(gy, gx), np.pi)[edges > 0]
    e_h = np.histogram(ang, bins=8, range=(0, np.pi))[0].astype(np.float64)
    e_h /= max(e_h.sum(), 1)
    f = np.concatenate([lbp_h, *col, e_h]).astype(np.float32)
    assert f.shape == (42,)
    return f


def load_features(data_dir, relpaths, cache_path):
    """Compute (or load cached) raw 42-D features for every relpath. Atomic write for parallel jobs."""
    from PIL import Image
    if os.path.exists(cache_path):
        z = np.load(cache_path, allow_pickle=False)
        cached = dict(zip(z["relpath"].tolist(), z["feat"]))
        if all(r in cached for r in relpaths):
            return cached
    log(f"computing handcrafted features for {len(relpaths)} images ...")
    feats = np.stack([handcrafted_42(np.array(Image.open(os.path.join(data_dir, r)).convert("RGB")))
                      for r in relpaths])
    tmp = f"{cache_path}.{os.getpid()}.tmp.npz"
    np.savez(tmp, relpath=np.array(relpaths), feat=feats)
    os.replace(tmp, cache_path)
    return dict(zip(relpaths, feats))


# ----------------------------------------------------------------------------------------------
# Model
# ----------------------------------------------------------------------------------------------
def build_model(variant, pretrained=True):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torchvision

    class CCDGS(nn.Module):
        """Channel-Centric Depth-wise Group Shuffle block (spatial maps preserved, residual)."""
        def __init__(self, c, groups):
            super().__init__()
            self.g = groups
            self.gconv = nn.Conv2d(c, c, 3, padding=1, groups=groups, bias=False); self.bn1 = nn.BatchNorm2d(c)
            self.dw = nn.Conv2d(c, c, 3, padding=1, groups=c, bias=False); self.bn2 = nn.BatchNorm2d(c)
            self.pw = nn.Conv2d(c, c, 1, bias=False); self.bn3 = nn.BatchNorm2d(c)

        def forward(self, x):
            y = F.relu(self.bn1(self.gconv(x)))
            b, c, h, w = y.shape
            y = y.view(b, self.g, c // self.g, h, w).transpose(1, 2).contiguous().view(b, c, h, w)  # shuffle
            y = F.relu(self.bn2(self.dw(y)))
            y = self.bn3(self.pw(y))
            return F.relu(x + y)

    class ZPool(nn.Module):
        def forward(self, x):
            return torch.cat([x.max(1, keepdim=True)[0], x.mean(1, keepdim=True)], 1)

    class Gate(nn.Module):
        def __init__(self, k):
            super().__init__()
            self.z = ZPool(); self.conv = nn.Conv2d(2, 1, k, padding=k // 2, bias=False); self.bn = nn.BatchNorm2d(1)

        def forward(self, x):
            return x * torch.sigmoid(self.bn(self.conv(self.z(x))))

    class TripletAttention(nn.Module):
        """Misra et al. (WACV 2021): C-H, C-W and H-W interaction branches, averaged."""
        def __init__(self, k):
            super().__init__()
            self.ch, self.cw, self.hw = Gate(k), Gate(k), Gate(k)

        def forward(self, x):
            o1 = self.ch(x.permute(0, 2, 1, 3).contiguous()).permute(0, 2, 1, 3)
            o2 = self.cw(x.permute(0, 3, 2, 1).contiguous()).permute(0, 3, 2, 1)
            o3 = self.hw(x)
            return (o1 + o2 + o3) / 3.0

    class SE(nn.Module):
        def __init__(self, c, r):
            super().__init__()
            self.fc = nn.Sequential(nn.Linear(c, c // r, bias=False), nn.ReLU(inplace=True),
                                    nn.Linear(c // r, c, bias=False), nn.Sigmoid())

        def forward(self, x):
            return x * self.fc(x.mean((2, 3)))[:, :, None, None]

    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            w = torchvision.models.DenseNet169_Weights.IMAGENET1K_V1 if pretrained else None
            self.features = torchvision.models.densenet169(weights=w).features
            self.ccdgs = CCDGS(1664, CFG["ccdgs_groups"])
            self.ta = TripletAttention(CFG["ta_kernel"])
            self.se = SE(1664, CFG["se_reduction"]) if USES_SE[variant] else nn.Identity()
            self.use_hc = USES_HC[variant]
            fin = 1664 + (CFG["hc_dim"] if self.use_hc else 0)
            self.head = nn.Sequential(nn.Dropout(CFG["dropout"]), nn.Linear(fin, CFG["hidden"]), nn.ReLU(inplace=True),
                                      nn.Dropout(CFG["dropout"]), nn.Linear(CFG["hidden"], NC))

        def forward(self, x, hc=None):
            f = F.relu(self.features(x))
            f = self.se(self.ta(self.ccdgs(f)))
            f = f.mean((2, 3))
            if self.use_hc:
                assert hc is not None and hc.shape[1] == CFG["hc_dim"]
                f = torch.cat([f, hc], 1)
            return self.head(f)

    assert variant in VARIANTS
    return Net()


# ----------------------------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------------------------
def read_split(split_dir, limit_per_class=None):
    sp = {}
    for n in ["train", "validation", "test"]:
        d = pd.read_csv(os.path.join(split_dir, f"{n}.csv"))
        if limit_per_class:
            d = d.groupby("label", group_keys=False).head(limit_per_class).reset_index(drop=True)
        sp[n] = d
    for k in ["relpath", "md5", "group_id"]:
        s = {n: set(d[k]) for n, d in sp.items()}
        assert not (s["train"] & s["validation"]) and not (s["train"] & s["test"]) and not (s["validation"] & s["test"]), \
            f"leakage: {k} shared between subsets"
    return sp


def make_loaders(sp, data_dir, feats, seed, batch, workers, img_size):
    import torch
    from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
    import torchvision.transforms as T
    from PIL import Image

    norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    train_tf = T.Compose([T.RandomResizedCrop(img_size, scale=(0.8, 1.0)), T.RandomHorizontalFlip(),
                          T.RandomVerticalFlip(), T.RandomRotation(15),
                          T.ColorJitter(0.2, 0.2, 0.2, 0.02), T.ToTensor(), norm])
    eval_tf = T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), norm])

    tr = np.stack([feats[r] for r in sp["train"].relpath])
    mu, sd = tr.mean(0), tr.std(0)
    sd[sd < 1e-6] = 1.0  # standardisation fitted on TRAIN only

    class DS(Dataset):
        def __init__(self, d, tf):
            self.p = d.relpath.tolist(); self.y = d.label.astype(int).tolist(); self.tf = tf
            self.h = [((feats[r] - mu) / sd).astype(np.float32) for r in self.p]

        def __len__(self):
            return len(self.p)

        def __getitem__(self, i):
            # transform is chosen by SPLIT only, never by label
            img = Image.open(os.path.join(data_dir, self.p[i])).convert("RGB")
            return self.tf(img), self.y[i], torch.from_numpy(self.h[i])

    y = sp["train"].label.values
    cw = 1.0 / np.bincount(y, minlength=NC)
    g = torch.Generator(); g.manual_seed(seed)
    sampler = WeightedRandomSampler(torch.as_tensor(cw[y], dtype=torch.double), len(y), replacement=True, generator=g)

    def wif(wid):
        s = torch.initial_seed() % 2 ** 32
        np.random.seed(s); random.seed(s)

    kw = dict(batch_size=batch, num_workers=workers, worker_init_fn=wif, pin_memory=torch.cuda.is_available())
    dl = dict(train=DataLoader(DS(sp["train"], train_tf), sampler=sampler, **kw),
              validation=DataLoader(DS(sp["validation"], eval_tf), shuffle=False, **kw),
              test=DataLoader(DS(sp["test"], eval_tf), shuffle=False, **kw))
    return dl, g, (train_tf, eval_tf)


def assert_eval_transform_ok(eval_tf):
    import torchvision.transforms as T
    rnd = (T.RandomResizedCrop, T.RandomHorizontalFlip, T.RandomVerticalFlip, T.RandomRotation, T.ColorJitter,
           T.RandomAffine, T.RandomCrop, T.RandomErasing, T.RandomApply, T.RandomChoice)
    assert not any(isinstance(t, rnd) for t in eval_tf.transforms), "eval transform contains random ops"


# ----------------------------------------------------------------------------------------------
# Metrics (fast numpy versions; used in training and bootstrap)
# ----------------------------------------------------------------------------------------------
def confusion(y, p):
    return np.bincount(y * NC + p, minlength=NC * NC).reshape(NC, NC)


def macro_f1(y, p):
    cm = confusion(y, p).astype(float)
    tp = np.diag(cm); prec_d = cm.sum(0); rec_d = cm.sum(1)
    f1 = np.where(prec_d + rec_d > 0, 2 * tp / np.maximum(prec_d + rec_d, 1e-12), 0.0)
    return float(f1.mean())


# ----------------------------------------------------------------------------------------------
# TRAIN
# ----------------------------------------------------------------------------------------------
def cmd_train(a):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    t_start = time.time()
    os.makedirs(a.out_dir, exist_ok=True)
    done_flag = os.path.join(a.out_dir, "DONE")
    if os.path.exists(done_flag):
        log(f"{a.out_dir} already DONE - nothing to do"); return 0

    smoke = a.smoke
    if smoke:
        CFG.update(max_epochs=a.smoke_epochs, early_stop_patience=99, batch_size=a.smoke_batch)
    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed); torch.cuda.manual_seed_all(a.seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = dev.type == "cuda"

    sp = read_split(a.split_dir, limit_per_class=16 if smoke else None)
    allrel = sorted(pd.concat(sp.values()).relpath)
    cache = os.path.join(os.path.dirname(os.path.abspath(a.split_dir)), "features42_cache.npz")  # next to split/, not inside
    feats = load_features(a.data_dir, allrel, cache if not smoke else os.path.join(a.out_dir, "features42_smoke.npz"))
    dl, gen, (train_tf, eval_tf) = make_loaders(sp, a.data_dir, feats, a.seed, CFG["batch_size"], a.workers, CFG["img_size"])
    assert_eval_transform_ok(eval_tf)

    model = build_model(a.variant, pretrained=not a.no_pretrained).to(dev)
    if not a.no_pretrained:
        # ImageNet weights check: first conv must equal the torchvision ImageNet tensor
        import torchvision
        ref = torchvision.models.densenet169(weights=torchvision.models.DenseNet169_Weights.IMAGENET1K_V1).features.conv0.weight
        assert torch.allclose(model.features.conv0.weight.detach().cpu(), ref), "ImageNet weights not loaded"
    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=5, min_lr=1e-7)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    def focal(logits, y, gamma=2.0):
        ce = F.cross_entropy(logits.float(), y, reduction="none")
        return ((1 - torch.exp(-ce)) ** gamma * ce).mean()

    last_p, best_p = os.path.join(a.out_dir, "last.pth"), os.path.join(a.out_dir, "best.pth")
    st = dict(epoch=0, best_f1=-1.0, best_epoch=-1, bad=0, history=[])
    if os.path.exists(last_p):
        ck = torch.load(last_p, map_location=dev, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"]); st = ck["state"]; gen.set_state(ck["gen"])
        torch.set_rng_state(ck["rng_torch"]); np.random.set_state(ck["rng_np"]); random.setstate(ck["rng_py"])
        if use_amp and ck.get("rng_cuda") is not None:
            torch.cuda.set_rng_state_all(ck["rng_cuda"])
        log(f"RESUMED from epoch {st['epoch']} (best macro-F1 {st['best_f1']:.4f} @ epoch {st['best_epoch']})")

    meta = dict(variant=a.variant, seed=a.seed, smoke=smoke, pretrained=not a.no_pretrained, cfg=CFG,
                split_md5={n: md5_file(os.path.join(a.split_dir, f"{n}.csv")) for n in ["train", "validation", "test"]},
                script_md5=md5_file(os.path.abspath(__file__)), python=platform.python_version(),
                torch=torch.__version__, cuda=torch.version.cuda,
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
                n=dict((k, len(v)) for k, v in sp.items()),
                params=sum(p.numel() for p in model.parameters()))
    json.dump(meta, open(os.path.join(a.out_dir, "run_meta.json"), "w"), indent=2)
    log(f"variant={a.variant} seed={a.seed} device={meta['gpu']} params={meta['params']:,} n={meta['n']}")

    def run_eval(loader):
        model.eval(); P, Y, L = [], [], 0.0
        with torch.no_grad():
            for x, y, h in loader:
                x, y, h = x.to(dev, non_blocking=True), y.to(dev), h.to(dev)
                with torch.autocast(device_type=dev.type, enabled=use_amp):
                    out = model(x, h)
                L += focal(out, y).item() * len(y)
                P.append(torch.softmax(out.float(), 1).cpu().numpy()); Y.append(y.cpu().numpy())
        P, Y = np.concatenate(P), np.concatenate(Y)
        return P, Y, L / len(Y)

    epoch_times = []
    while st["epoch"] < CFG["max_epochs"] and st["bad"] < CFG["early_stop_patience"]:
        t0 = time.time(); model.train(); tl, tc, tn = 0.0, 0, 0
        for x, y, h in dl["train"]:
            x, y, h = x.to(dev, non_blocking=True), y.to(dev), h.to(dev)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=dev.type, enabled=use_amp):
                out = model(x, h)
            loss = focal(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); tn += len(y)
        Pv, Yv, vl = run_eval(dl["validation"])
        vf1 = macro_f1(Yv, Pv.argmax(1)); vacc = float((Pv.argmax(1) == Yv).mean())
        sched.step(vf1)
        st["epoch"] += 1
        improved = vf1 > st["best_f1"]
        if improved:
            st.update(best_f1=vf1, best_epoch=st["epoch"], bad=0)
            torch.save(dict(model=model.state_dict(), epoch=st["epoch"], val_macro_f1=vf1), best_p)
        else:
            st["bad"] += 1
        st["history"].append(dict(epoch=st["epoch"], train_loss=tl / tn, train_acc=tc / tn, val_loss=vl,
                                  val_acc=vacc, val_macro_f1=vf1, lr=opt.param_groups[0]["lr"]))
        pd.DataFrame(st["history"]).to_csv(os.path.join(a.out_dir, "history.csv"), index=False)
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        scaler=scaler.state_dict(), state=st, gen=gen.get_state(),
                        rng_torch=torch.get_rng_state(), rng_np=np.random.get_state(), rng_py=random.getstate(),
                        rng_cuda=torch.cuda.get_rng_state_all() if use_amp else None), last_p + ".tmp")
        os.replace(last_p + ".tmp", last_p)
        epoch_times.append(time.time() - t0)
        log(f"ep {st['epoch']:3d} | train loss {tl/tn:.4f} acc {tc/tn:.4f} | val loss {vl:.4f} acc {vacc:.4f} "
            f"macroF1 {vf1:.4f}{' *best*' if improved else ''} | lr {opt.param_groups[0]['lr']:.1e} | {epoch_times[-1]:.0f}s"
            + (f" | peak GPU mem {torch.cuda.max_memory_allocated() / 1e9:.1f} GB" if use_amp else ""))
        elapsed_h = (time.time() - t_start) / 3600
        if elapsed_h + 1.5 * max(epoch_times) / 3600 > a.max_hours:
            log(f"time guard: {elapsed_h:.2f} h used of {a.max_hours} h - state saved, exiting for resubmission")
            return 0

    log(f"training finished: epochs {st['epoch']}, best val macro-F1 {st['best_f1']:.4f} @ epoch {st['best_epoch']}")
    ck = torch.load(best_p, map_location=dev, weights_only=False)
    model.load_state_dict(ck["model"])
    for n in ["validation", "test"]:
        P, Y, _ = run_eval(dl[n])
        d = sp[n][["relpath", "group_id"]].copy()
        d["true"] = Y; d["pred"] = P.argmax(1)
        for i, c in enumerate(CLASSES):
            d[f"prob_{c}"] = P[:, i]
        d.to_csv(os.path.join(a.out_dir, f"{'val' if n == 'validation' else n}_predictions.csv"), index=False)
        log(f"{n}: acc {(d.true == d.pred).mean():.4f} macroF1 {macro_f1(d.true.values, d.pred.values):.4f} "
            f"(best checkpoint, epoch {ck['epoch']})")
    meta.update(best_epoch=st["best_epoch"], best_val_macro_f1=st["best_f1"], epochs_run=st["epoch"],
                best_ckpt_md5=md5_file(best_p))
    json.dump(meta, open(os.path.join(a.out_dir, "run_meta.json"), "w"), indent=2)
    open(done_flag, "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))
    if os.path.exists(last_p) and not a.keep_last:
        os.remove(last_p)
    return 0


# ----------------------------------------------------------------------------------------------
# EVAL
# ----------------------------------------------------------------------------------------------
def full_metrics(y, pred, prob):
    from sklearn.metrics import roc_auc_score, f1_score
    cm = confusion(y, pred)
    r = dict(n=len(y), accuracy=float((y == pred).mean()), macro_f1=macro_f1(y, pred),
             weighted_f1=float(f1_score(y, pred, average="weighted", labels=list(range(NC)), zero_division=0)))
    try:
        r["macro_auc"] = float(roc_auc_score(y, prob, multi_class="ovr", average="macro", labels=list(range(NC))))
    except ValueError:
        r["macro_auc"] = float("nan")
    for i, c in enumerate(CLASSES):
        tp, fp, fn = cm[i, i], cm[:, i].sum() - cm[i, i], cm[i, :].sum() - cm[i, i]
        p = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        r[f"{c}_precision"] = float(p); r[f"{c}_recall"] = float(rc)
        r[f"{c}_f1"] = float(2 * p * rc / (p + rc)) if p + rc else 0.0
        r[f"{c}_correct"] = int(tp); r[f"{c}_total"] = int(cm[i, :].sum())
    return r


def hard_vote(preds, probs):
    """Majority vote; 3-way tie -> class with highest mean softmax probability."""
    preds = np.stack(preds, 1); mp = np.mean(probs, 0)
    out, ties = np.empty(len(preds), int), 0
    for i, row in enumerate(preds):
        cnt = np.bincount(row, minlength=NC)
        top = np.flatnonzero(cnt == cnt.max())
        if len(top) > 1:
            ties += 1
            out[i] = top[np.argmax(mp[i, top])]
        else:
            out[i] = top[0]
    return out, mp, ties


def holm(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p); adj = np.empty(m)
    run = 0.0
    for k, i in enumerate(order):
        run = max(run, min(1.0, (m - k) * p[i])); adj[i] = run
    return adj


def cmd_eval(a):
    from scipy.stats import binomtest
    os.makedirs(a.out_dir, exist_ok=True)
    seeds = a.seeds
    runs, ref = {}, None
    for v in ["meff", "dsg"]:
        for s in seeds:
            rd = os.path.join(a.runs_dir, f"{v}_s{s}")
            if not os.path.exists(os.path.join(rd, "DONE")):
                sys.exit(f"ERROR: {rd} is not DONE - finish or resume that run first")
            meta = json.load(open(os.path.join(rd, "run_meta.json")))
            if (meta["smoke"] or not meta["pretrained"]) and not a.allow_smoke:
                sys.exit(f"ERROR: {rd} is a smoke/no-pretrained run - not valid for results")
            te = pd.read_csv(os.path.join(rd, "test_predictions.csv"))
            va = pd.read_csv(os.path.join(rd, "val_predictions.csv"))
            if ref is None:
                ref = te[["relpath", "group_id", "true"]]
            assert te.relpath.equals(ref.relpath) and te.true.equals(ref.true), f"{rd}: test rows differ"
            runs[(v, s)] = dict(te=te, va=va, meta=meta)
    y = ref.true.values; groups = ref.group_id.values
    pc = [f"prob_{c}" for c in CLASSES]
    rows, preds = [], {}
    for v in ["meff", "dsg"]:
        for s in seeds:
            te = runs[(v, s)]["te"]; nm = f"{v.upper()}_s{s}"
            preds[nm] = te.pred.values
            rows.append(dict(model=nm, **full_metrics(y, te.pred.values, te[pc].values)))
        seed_rows = pd.DataFrame([r for r in rows if r["model"].startswith(v.upper() + "_s")]).drop(columns="model")
        rows.append(dict(model=f"{v.upper()}_seed_mean", **seed_rows.mean().to_dict()))
        rows.append(dict(model=f"{v.upper()}_seed_sd", **seed_rows.std(ddof=1).to_dict()))
    ties = {}
    for v in ["meff", "dsg"]:
        ps = [runs[(v, s)]["te"].pred.values for s in seeds]
        pr = [runs[(v, s)]["te"][pc].values for s in seeds]
        hv, mp, nt = hard_vote(ps, pr); ties[v.upper()] = nt
        preds[f"{v.upper()}_hard"] = hv; preds[f"{v.upper()}_soft"] = mp.argmax(1)
        rows.append(dict(model=f"{v.upper()}_hard", **full_metrics(y, hv, mp)))
        rows.append(dict(model=f"{v.upper()}_soft", **full_metrics(y, mp.argmax(1), mp)))
        pd.DataFrame(confusion(y, hv), index=[f"true_{c}" for c in CLASSES],
                     columns=[f"pred_{c}" for c in CLASSES]).to_csv(os.path.join(a.out_dir, f"confusion_{v.upper()}_hard.csv"))
    res = pd.DataFrame(rows)
    res.to_csv(os.path.join(a.out_dir, "results.csv"), index=False)
    pd.DataFrame([dict(ensemble=k, three_way_ties=v, n=len(y)) for k, v in ties.items()]).to_csv(
        os.path.join(a.out_dir, "ensemble_ties.csv"), index=False)

    # best single MEFF seed chosen on VALIDATION macro-F1 (never on test)
    vf = {s: macro_f1(runs[("meff", s)]["va"].true.values, runs[("meff", s)]["va"].pred.values) for s in seeds}
    best_s = max(seeds, key=lambda s: vf[s])
    comps = [("MEFF_hard", "DSG_hard")] + [("MEFF_hard", f"DSG_s{s}") for s in seeds] + [("MEFF_hard", f"MEFF_s{best_s}")]

    # paired group-level bootstrap
    ug = np.unique(groups); idx_by_g = {g: np.flatnonzero(groups == g) for g in ug}
    rng = np.random.default_rng(42)
    samples = [np.concatenate([idx_by_g[g] for g in rng.choice(ug, len(ug), replace=True)]) for _ in range(a.n_boot)]
    out = []
    for A, B in comps:
        pa, pb = preds[A], preds[B]
        df1 = [macro_f1(y[i], pa[i]) - macro_f1(y[i], pb[i]) for i in samples]
        dac = [(y[i] == pa[i]).mean() - (y[i] == pb[i]).mean() for i in samples]
        ca, cb = pa == y, pb == y
        b, c = int((ca & ~cb).sum()), int((~ca & cb).sum())
        p = binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0
        out.append(dict(A=A, B=B, delta_macro_f1=macro_f1(y, pa) - macro_f1(y, pb),
                        ci95_low_macro_f1=float(np.percentile(df1, 2.5)), ci95_high_macro_f1=float(np.percentile(df1, 97.5)),
                        delta_accuracy=float(ca.mean() - cb.mean()),
                        ci95_low_accuracy=float(np.percentile(dac, 2.5)), ci95_high_accuracy=float(np.percentile(dac, 97.5)),
                        mcnemar_b_A_right_B_wrong=b, mcnemar_c_A_wrong_B_right=c, mcnemar_p_exact=float(p)))
    comp = pd.DataFrame(out); comp["mcnemar_p_holm"] = holm(comp.mcnemar_p_exact.values)
    comp["best_meff_seed_by_val_macro_f1"] = best_s
    comp.to_csv(os.path.join(a.out_dir, "comparison.csv"), index=False)

    prim = comp.iloc[0]
    if prim.ci95_low_macro_f1 > 0:
        verdict = "MEFF outperforms Dense-ShuffleGCANet"
    elif prim.ci95_high_macro_f1 < 0:
        verdict = "Dense-ShuffleGCANet outperforms MEFF"
    else:
        verdict = "No significant difference between MEFF and Dense-ShuffleGCANet"
    txt = (f"Primary comparison: MEFF_hard - DSG_hard, test macro-F1\n"
           f"delta = {prim.delta_macro_f1:.6f}, 95% CI [{prim.ci95_low_macro_f1:.6f}, {prim.ci95_high_macro_f1:.6f}] "
           f"(paired group bootstrap, {a.n_boot} resamples, seed 42, {len(ug)} test groups)\n"
           f"Rule: CI entirely > 0 -> MEFF outperforms; entirely < 0 -> DSG outperforms; else no significant difference.\n"
           f"DECISION: {verdict}\n")
    open(os.path.join(a.out_dir, "decision.txt"), "w").write(txt)
    prot = os.path.join(os.path.dirname(os.path.abspath(a.split_dir)), "PROTOCOL_v3.md")
    rec = dict(protocol_md5=md5_file(prot) if os.path.exists(prot) else None, script_md5=md5_file(os.path.abspath(__file__)),
               runs={f"{v}_s{s}": runs[(v, s)]["meta"].get("best_ckpt_md5") for v in ["meff", "dsg"] for s in seeds})
    json.dump(rec, open(os.path.join(a.out_dir, "eval_meta.json"), "w"), indent=2)
    pd.set_option("display.width", 200)
    print(res[["model", "n", "accuracy", "macro_f1", "weighted_f1", "macro_auc", "ISCHAEMIA_recall", "BOTH_recall"]].round(4).to_string(index=False))
    print(comp[["A", "B", "delta_macro_f1", "ci95_low_macro_f1", "ci95_high_macro_f1", "mcnemar_p_holm"]].round(4).to_string(index=False))
    print(txt)
    return 0


# ----------------------------------------------------------------------------------------------
# VERIFY: dataset on disk must match the frozen split (every file present, MD5 identical, nothing extra)
# ----------------------------------------------------------------------------------------------
def cmd_verify(a):
    sp = read_split(a.split_dir)
    allr = pd.concat(sp.values())
    on_disk = {os.path.relpath(os.path.join(r, f), a.data_dir) for r, _, fs in os.walk(a.data_dir)
               for f in fs if f.lower().endswith((".jpg", ".jpeg", ".png"))}
    missing = [r for r in allr.relpath if r not in on_disk]
    extra = sorted(on_disk - set(allr.relpath))
    bad = [r for r, m in zip(allr.relpath, allr.md5) if r in on_disk and md5_file(os.path.join(a.data_dir, r)) != m]
    print(f"split rows {len(allr)} | files on disk {len(on_disk)} | missing {len(missing)} | extra {len(extra)} | md5 mismatch {len(bad)}")
    for n, d in sp.items():
        print(f"  {n:10s} {len(d):5d}  " + "  ".join(f"{c}={int((d.class_name == c).sum())}" for c in CLASSES))
    man = os.path.join(a.split_dir, "SPLIT_MANIFEST.json")
    if os.path.exists(man):
        exp = json.load(open(man))["files_md5"]
        for f, m in exp.items():
            ok = md5_file(os.path.join(a.split_dir, f)) == m
            print(f"  {f:16s} md5 {'OK' if ok else 'CHANGED'}")
            bad += [] if ok else [f]
    if missing or extra or bad or len(allr) != 5955:
        print("VERIFY FAILED - do not train"); return 1
    print("VERIFY OK"); return 0


# ----------------------------------------------------------------------------------------------
# COMPLEXITY
# ----------------------------------------------------------------------------------------------
def cmd_complexity(a):
    import torch
    from torch.utils.flop_counter import FlopCounterMode
    os.makedirs(a.out_dir, exist_ok=True)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    rows = []
    for v in VARIANTS:
        m = build_model(v, pretrained=False).to(dev).eval()
        x = torch.randn(1, 3, 224, 224, device=dev); h = torch.randn(1, 42, device=dev)
        with torch.no_grad():
            fc = FlopCounterMode(display=False)
            with fc:
                m(x, h)
            flops = fc.get_total_flops()
            for _ in range(5):
                m(x, h)
            if dev.type == "cuda":
                torch.cuda.synchronize()
            ts = []
            for _ in range(a.n_timing):
                t0 = time.perf_counter(); m(x, h)
                if dev.type == "cuda":
                    torch.cuda.synchronize()
                ts.append(time.perf_counter() - t0)
        prm = sum(p.numel() for p in m.parameters())
        rows.append(dict(model={"meff": "MEFF expert", "dsg": "Dense-ShuffleGCANet (DSG)", "dsg_se": "DSG + SE",
                                "dsg_hc": "DSG + handcrafted"}[v], params=prm,
                         params_M=prm / 1e6, gflops=flops / 1e9, gmacs=flops / 2e9,
                         latency_ms_batch1_median=1000 * float(np.median(ts)), device=str(dev)))
        if v == "meff":
            rows.append(dict(model="MEFF ensemble (3 experts)", params=3 * prm, params_M=3 * prm / 1e6,
                             gflops=3 * flops / 1e9, gmacs=3 * flops / 2e9,
                             latency_ms_batch1_median=3000 * float(np.median(ts)), device=str(dev)))
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(a.out_dir, "complexity_v3.csv"), index=False)
    print(df.round(3).to_string(index=False))
    return 0


# ----------------------------------------------------------------------------------------------
# ABLATION (PROTOCOL_ABLATION.md) - SE x handcrafted, 2x2, 3 seeds per cell
# ----------------------------------------------------------------------------------------------
ABL_COMPS = [("DSG_SE_hard", "DSG_hard", "A1: effect of SE (without handcrafted)"),
             ("DSG_HC_hard", "DSG_hard", "A2: effect of handcrafted (without SE)"),
             ("MEFF_hard", "DSG_SE_hard", "A3: effect of handcrafted (with SE)"),
             ("MEFF_hard", "DSG_HC_hard", "A4: effect of SE (with handcrafted)")]


def anova_2x2(cells):
    """Balanced two-way ANOVA. cells[(se, hc)] = list of n replicate values."""
    from scipy.stats import f as fdist
    n = len(next(iter(cells.values()))); assert all(len(v) == n for v in cells.values())
    m = {k: np.mean(v) for k, v in cells.items()}; g = np.mean([x for v in cells.values() for x in v])
    mA = {i: np.mean([m[(i, j)] for j in (0, 1)]) for i in (0, 1)}; mB = {j: np.mean([m[(i, j)] for i in (0, 1)]) for j in (0, 1)}
    ss = dict(SE=2 * n * sum((mA[i] - g) ** 2 for i in (0, 1)), handcrafted=2 * n * sum((mB[j] - g) ** 2 for j in (0, 1)),
              interaction=n * sum((m[(i, j)] - mA[i] - mB[j] + g) ** 2 for i in (0, 1) for j in (0, 1)))
    sse = sum((x - m[k]) ** 2 for k, v in cells.items() for x in v); dfe = 4 * (n - 1); mse = sse / dfe
    rows = [dict(effect=k, SS=v, df=1, F=v / mse, p=float(fdist.sf(v / mse, 1, dfe))) for k, v in ss.items()]
    rows.append(dict(effect="residual", SS=sse, df=dfe, F=np.nan, p=np.nan))
    return pd.DataFrame(rows)


def cmd_ablation(a):
    from scipy.stats import binomtest
    os.makedirs(a.out_dir, exist_ok=True)
    seeds = a.seeds; runs, ref, cfg0, split0 = {}, None, None, None
    for v in VARIANTS:
        for s in seeds:
            rd = os.path.join(a.runs_dir, f"{v}_s{s}")
            if not os.path.exists(os.path.join(rd, "DONE")):
                sys.exit(f"ERROR: {rd} is not DONE")
            meta = json.load(open(os.path.join(rd, "run_meta.json")))
            if (meta["smoke"] or not meta["pretrained"]) and not a.allow_smoke:
                sys.exit(f"ERROR: {rd} is a smoke/no-pretrained run")
            assert meta["variant"] == v and meta["seed"] == s, f"{rd}: run_meta says {meta['variant']} seed {meta['seed']}"
            cfg = {k: meta["cfg"][k] for k in meta["cfg"] if k not in ("max_epochs", "early_stop_patience", "batch_size")} \
                if a.allow_smoke else meta["cfg"]
            cfg0 = cfg0 or cfg; split0 = split0 or meta["split_md5"]
            assert cfg == cfg0, f"{rd}: training recipe differs from the other runs"
            assert meta["split_md5"] == split0, f"{rd}: different split files"
            te = pd.read_csv(os.path.join(rd, "test_predictions.csv"))
            if ref is None:
                ref = te[["relpath", "group_id", "true"]]
            assert te.relpath.equals(ref.relpath) and te.true.equals(ref.true), f"{rd}: test rows differ"
            runs[(v, s)] = dict(te=te, meta=meta)
    y = ref.true.values; pc = [f"prob_{c}" for c in CLASSES]; rows, preds = [], {}
    for v in VARIANTS:
        V = v.upper()
        for s in seeds:
            te = runs[(v, s)]["te"]; preds[f"{V}_s{s}"] = te.pred.values
            rows.append(dict(model=f"{V}_s{s}", variant=v, SE=USES_SE[v], handcrafted=USES_HC[v], seed=s,
                             **full_metrics(y, te.pred.values, te[pc].values)))
        sr = pd.DataFrame([r for r in rows if r.get("variant") == v and r.get("seed") is not None]).drop(
            columns=["model", "variant", "SE", "handcrafted", "seed"])
        rows.append(dict(model=f"{V}_seed_mean", variant=v, **sr.mean().to_dict()))
        rows.append(dict(model=f"{V}_seed_sd", variant=v, **sr.std(ddof=1).to_dict()))
        hv, mp, _ = hard_vote([runs[(v, s)]["te"].pred.values for s in seeds], [runs[(v, s)]["te"][pc].values for s in seeds])
        preds[f"{V}_hard"] = hv; preds[f"{V}_soft"] = mp.argmax(1)
        rows.append(dict(model=f"{V}_hard", variant=v, SE=USES_SE[v], handcrafted=USES_HC[v], **full_metrics(y, hv, mp)))
        rows.append(dict(model=f"{V}_soft", variant=v, SE=USES_SE[v], handcrafted=USES_HC[v], **full_metrics(y, mp.argmax(1), mp)))
    res = pd.DataFrame(rows); res.to_csv(os.path.join(a.out_dir, "ablation_results.csv"), index=False)

    rng = np.random.default_rng(42); n = len(y)
    samples = [rng.integers(0, n, n) for _ in range(a.n_boot)]   # image-level (one group per image in this split)
    out = []
    for A, B, lab in ABL_COMPS:
        pa, pb = preds[A], preds[B]
        d = [macro_f1(y[i], pa[i]) - macro_f1(y[i], pb[i]) for i in samples]
        ca, cb = pa == y, pb == y; b_, c_ = int((ca & ~cb).sum()), int((~ca & cb).sum())
        out.append(dict(comparison=lab, A=A, B=B, delta_macro_f1=macro_f1(y, pa) - macro_f1(y, pb),
                        ci95_low=float(np.percentile(d, 2.5)), ci95_high=float(np.percentile(d, 97.5)),
                        delta_accuracy=float(ca.mean() - cb.mean()), mcnemar_b=b_, mcnemar_c=c_,
                        mcnemar_p_exact=float(binomtest(min(b_, c_), b_ + c_, 0.5).pvalue) if b_ + c_ else 1.0))
    comp = pd.DataFrame(out); comp["mcnemar_p_holm"] = holm(comp.mcnemar_p_exact.values)
    comp["verdict"] = ["helps" if r.ci95_low > 0 else "hurts" if r.ci95_high < 0 else "no significant difference" for r in comp.itertuples()]
    comp.to_csv(os.path.join(a.out_dir, "ablation_comparison.csv"), index=False)

    cells = {(int(USES_SE[v]), int(USES_HC[v])): [macro_f1(y, preds[f"{v.upper()}_s{s}"]) for s in seeds] for v in VARIANTS}
    an = anova_2x2(cells); an.to_csv(os.path.join(a.out_dir, "ablation_anova.csv"), index=False)

    def component(name, rows_):
        v = list(comp.verdict.iloc[rows_])
        return f"{name}: " + ("HELPS (CI > 0 in both comparisons)" if v == ["helps"] * 2 else
                              "HURTS (CI < 0 in both comparisons)" if v == ["hurts"] * 2 else
                              f"NO CONSISTENT SIGNIFICANT EFFECT ({v[0]}; {v[1]})")
    txt = ("Pre-registered ablation (PROTOCOL_ABLATION.md): hard-vote ensembles, test macro-F1, paired bootstrap "
           f"({a.n_boot} resamples, seed 42), McNemar exact + Holm over A1-A4.\n"
           + "\n".join(f"{r.comparison}: delta {r.delta_macro_f1:+.4f}, 95% CI [{r.ci95_low:+.4f}, {r.ci95_high:+.4f}] -> {r.verdict}"
                       for r in comp.itertuples())
           + "\n" + component("SE block", [0, 3]) + "\n" + component("Handcrafted features", [1, 2])
           + "\nSecondary: two-way ANOVA on single-model test macro-F1 (3 seeds per cell): "
           + "; ".join(f"{r.effect} p={r.p:.4f}" for r in an.itertuples() if r.effect != "residual") + "\n")
    open(os.path.join(a.out_dir, "ablation_decision.txt"), "w").write(txt)
    prot = os.path.join(os.path.dirname(os.path.abspath(__file__)), "PROTOCOL_ABLATION.md")
    json.dump(dict(code_md5=md5_file(os.path.abspath(__file__)), protocol_md5=md5_file(prot) if os.path.exists(prot) else None,
                   run_script_md5={f"{v}_s{s}": runs[(v, s)]["meta"]["script_md5"] for v in VARIANTS for s in seeds},
                   run_ckpt_md5={f"{v}_s{s}": runs[(v, s)]["meta"].get("best_ckpt_md5") for v in VARIANTS for s in seeds}),
              open(os.path.join(a.out_dir, "ablation_meta.json"), "w"), indent=2)
    pd.set_option("display.width", 220)
    print(res[res.model.str.endswith(("_hard", "_seed_mean", "_seed_sd"))][["model", "accuracy", "macro_f1", "macro_auc"]].round(4).to_string(index=False))
    print(comp[["comparison", "delta_macro_f1", "ci95_low", "ci95_high", "mcnemar_p_holm", "verdict"]].round(4).to_string(index=False))
    print(an.round(5).to_string(index=False)); print(txt)
    return 0


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    t = sub.add_parser("train")
    t.add_argument("--variant", choices=VARIANTS, required=True)
    t.add_argument("--seed", type=int, required=True)
    t.add_argument("--data_dir", required=True)
    t.add_argument("--split_dir", required=True)
    t.add_argument("--out_dir", required=True)
    t.add_argument("--workers", type=int, default=4)
    t.add_argument("--max_hours", type=float, default=46.0, help="stop cleanly before this many hours")
    t.add_argument("--keep_last", action="store_true", help="keep last.pth after DONE")
    t.add_argument("--no_pretrained", action="store_true", help="SMOKE TESTS ONLY")
    t.add_argument("--smoke", action="store_true", help="16 images/class/split, few epochs - pipeline test only")
    t.add_argument("--smoke_epochs", type=int, default=2)
    t.add_argument("--smoke_batch", type=int, default=4, help="batch size in smoke mode only (real runs: 32)")
    e = sub.add_parser("eval")
    e.add_argument("--runs_dir", required=True)
    e.add_argument("--split_dir", required=True)
    e.add_argument("--out_dir", required=True)
    e.add_argument("--seeds", type=int, nargs="+", default=[42, 43, 44])
    e.add_argument("--n_boot", type=int, default=2000)
    e.add_argument("--allow_smoke", action="store_true")
    v = sub.add_parser("verify")
    v.add_argument("--data_dir", required=True)
    v.add_argument("--split_dir", required=True)
    c = sub.add_parser("complexity")
    c.add_argument("--out_dir", required=True)
    c.add_argument("--n_timing", type=int, default=50)
    ab = sub.add_parser("ablation")
    ab.add_argument("--runs_dir", required=True)
    ab.add_argument("--out_dir", required=True)
    ab.add_argument("--seeds", type=int, nargs="+", default=[42, 43, 44])
    ab.add_argument("--n_boot", type=int, default=2000)
    ab.add_argument("--allow_smoke", action="store_true")
    a = ap.parse_args()
    return dict(train=cmd_train, eval=cmd_eval, verify=cmd_verify, complexity=cmd_complexity, ablation=cmd_ablation)[a.cmd](a)


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
md5 = hashlib.md5(open(f"{W}/meff_v3_1.py", "rb").read()).hexdigest()
print("meff_v3_1.py md5:", md5)
assert md5 == "77d87ea56c3e4585e89058a9952154c7", "meff_v3_1.py differs from the file used for the runs"


## 2. Test split (only used to pick the 200 timed images)

In [ ]:
%%writefile /kaggle/working/latency/test.csv
relpath,filename,class_name,label,md5,group_id
NONE/301001.jpg,301001.jpg,NONE,0,2a102026ab8b75e10f2055d0119a5203,1
NONE/301007.jpg,301007.jpg,NONE,0,cf60b012ad2a3b1ebe1851f3aea76502,4
NONE/301018.jpg,301018.jpg,NONE,0,28396c1c59c0fd723c19cd6fb780512e,11
NONE/301028.jpg,301028.jpg,NONE,0,c3d4b6624fbbfe82f8f1399bd71cf53e,16
NONE/301035.jpg,301035.jpg,NONE,0,939874ada6d205586b42daa6d4b4e815,19
NONE/301045.jpg,301045.jpg,NONE,0,c27f5c6cb5a0e1644ce2e0f300b9cf7d,21
NONE/301072.jpg,301072.jpg,NONE,0,98a417ce191c7023d12db1f54d25f9a6,34
NONE/301074.jpg,301074.jpg,NONE,0,f7d40adbbdc7f9fe68da7023eba3ac85,35
NONE/301078.jpg,301078.jpg,NONE,0,3061dc14f1b6604d479e343e6d74344f,36
NONE/301083.jpg,301083.jpg,NONE,0,2dc0439f4a50d66a165b4c5c2838e442,38
NONE/301105.jpg,301105.jpg,NONE,0,cf37772cdb3b2659b0bc2fcec927c0ce,46
NONE/301117.jpg,301117.jpg,NONE,0,c1382aa65c570aaceb94729aab1f9b78,50
NONE/301150.jpg,301150.jpg,NONE,0,f3f17ca7276916ebaac55a841a878a68,64
NONE/301177.jpg,301177.jpg,NONE,0,1d57b22a3ac7a0fcb86ffaaaeec852f3,75
NONE/301210.jpg,301210.jpg,NONE,0,bb252e1cde24ce191bfb11f728146d12,89
NONE/301213.jpg,301213.jpg,NONE,0,6715f9284889163ae6c067881bbe15ec,90
NONE/301229.jpg,301229.jpg,NONE,0,ff8615fe0ab7374d8474fdf859dd9d19,95
NONE/301242.jpg,301242.jpg,NONE,0,204159d2f14d5bb2737f119c9bb75837,98
NONE/301254.jpg,301254.jpg,NONE,0,d761d99fe279a94c3460c31e72c1a2ad,103
NONE/301279.jpg,301279.jpg,NONE,0,af2c5e1e995e47f29854fef385223bbf,114
NONE/301282.jpg,301282.jpg,NONE,0,1a2f30547a8aa2997e7054fcf3c08551,116
NONE/301289.jpg,301289.jpg,NONE,0,912aacd53b10a3dd80c8ea2ce1d3a281,119
NONE/301292.jpg,301292.jpg,NONE,0,1a041b149abca0e4783ec5dda2f80d60,122
NONE/301302.jpg,301302.jpg,NONE,0,e86fe4e9dac20633dff1a2cd29da6244,127
NONE/301306.jpg,301306.jpg,NONE,0,67285bd83a2b22e69fd326b4f7ef5aef,130
NONE/301332.jpg,301332.jpg,NONE,0,baabc99b144c9b9dfebd3a90e2650965,143
NONE/301336.jpg,301336.jpg,NONE,0,f5f101f09f3f7df52ef09461234b63e4,146
NONE/301354.jpg,301354.jpg,NONE,0,396e7e7928b605356c5274e8c87a0430,153
NONE/301357.jpg,301357.jpg,NONE,0,cb568a0b0844153f28b8b02a62804d0f,154
NONE/301371.jpg,301371.jpg,NONE,0,f98954036d6c8e736ad06c15850fb1db,160
NONE/301373.jpg,301373.jpg,NONE,0,0ee9290c54106176514cc12e776c3fce,161
NONE/301412.jpg,301412.jpg,NONE,0,ebf920a1a2600ff5bd8eb7e73af867f4,172
NONE/301428.jpg,301428.jpg,NONE,0,717792b5f3db07fc5cd91567aa4a0ee4,180
NONE/301440.jpg,301440.jpg,NONE,0,0104e2f06a12fe69c9b8c6fc66bd90c2,186
NONE/301448.jpg,301448.jpg,NONE,0,f52b1943fcceadf68c8df2156cf4056f,189
NONE/301449.jpg,301449.jpg,NONE,0,611ab3819ba6a1896b2e755482617d0e,190
NONE/301478.jpg,301478.jpg,NONE,0,5407f2a5f2f3852f9c880ee4be2d618a,197
NONE/301486.jpg,301486.jpg,NONE,0,9ef0d8d258c0a2f79676dd49a2f23d9a,200
NONE/301489.jpg,301489.jpg,NONE,0,603a568200af78f5504e6b2bfda615ef,201
NONE/301491.jpg,301491.jpg,NONE,0,7fe505a5b2bc1c626348e7a6ef978e22,202
NONE/301498.jpg,301498.jpg,NONE,0,889683f72792ed462e9e583053b4412b,206
NONE/301524.jpg,301524.jpg,NONE,0,ea3f9d7fc883e6ca86a979fc33dff45f,216
NONE/301541.jpg,301541.jpg,NONE,0,21e7bb1f525cad9eda4e7aea5cc704ea,225
NONE/301566.jpg,301566.jpg,NONE,0,ee765af7a7e18110de8bae8ca99448b6,234
NONE/301585.jpg,301585.jpg,NONE,0,fcc548b9ad66ce94ebfce531b369f0c6,241
NONE/301589.jpg,301589.jpg,NONE,0,1dfb4adf8a4edd1f431ffb69f1d0f086,245
NONE/301613.jpg,301613.jpg,NONE,0,65b7af589fabcbfe16583fa2fc3e3e72,253
NONE/301634.jpg,301634.jpg,NONE,0,6688c2c3c7d3c4bb1fadcc65cedd6864,262
NONE/301640.jpg,301640.jpg,NONE,0,3a78b4a25776491f15bc880e1a2e45a8,264
NONE/301654.jpg,301654.jpg,NONE,0,82c78c8a4b628a10d50ae0ce00b180e1,268
NONE/301696.jpg,301696.jpg,NONE,0,05c92a38e0da96d43ab616b35b6dd170,284
NONE/301723.jpg,301723.jpg,NONE,0,791525834a3894ff32a27ccf6ee46344,293
NONE/301737.jpg,301737.jpg,NONE,0,9da961d928b985351aa1450203e3d19e,301
NONE/301772.jpg,301772.jpg,NONE,0,8a468201ea324a95e03ceb87e8f59602,317
NONE/301804.jpg,301804.jpg,NONE,0,bbd17523415933a5429b65940b359e98,330
NONE/301822.jpg,301822.jpg,NONE,0,29d3ae895c0c13ca1471415824bdb25d,335
NONE/301824.jpg,301824.jpg,NONE,0,00d6ba1d0cef3f639b91dfee659ae0e7,337
NONE/301868.jpg,301868.jpg,NONE,0,f8b2778cb0bb086d4bac21dabcfbad7e,356
NONE/301898.jpg,301898.jpg,NONE,0,d60725902ea864e85c719a662107c6fc,369
NONE/301924.jpg,301924.jpg,NONE,0,83fbe2a5b63f36147c6531c3ee8986c7,378
NONE/301926.jpg,301926.jpg,NONE,0,e3be159017d0ddc8e173614998a78990,379
NONE/301943.jpg,301943.jpg,NONE,0,a42478793327f0e8793ece43be3b789d,386
NONE/301953.jpg,301953.jpg,NONE,0,0292d15b6f7cb77b7dac4b66f9386cfc,391
NONE/301954.jpg,301954.jpg,NONE,0,587e83b3301ae4159b36b0caf43a9074,392
NONE/301962.jpg,301962.jpg,NONE,0,81d1c53587f5f5b2753bbfcc507ff7e1,395
NONE/301966.jpg,301966.jpg,NONE,0,11f8e0882de9141d09c2c9a76ac40f3e,396
NONE/301969.jpg,301969.jpg,NONE,0,0e7b9564af0b44cfdf2d798c69f4cbff,397
NONE/301971.jpg,301971.jpg,NONE,0,e7f6945ad75ddef39e7665d6e157482a,399
NONE/301972.jpg,301972.jpg,NONE,0,75e3ee9c4f1eb71cad48e159117f733e,400
NONE/301975.jpg,301975.jpg,NONE,0,9199f46711721cc5751ffd8d509b7d21,403
NONE/301993.jpg,301993.jpg,NONE,0,4879ea3d43f2c914983e873192926d77,412
NONE/302004.jpg,302004.jpg,NONE,0,d626aa97ef26eb364a3cf83b95676d80,417
NONE/302007.jpg,302007.jpg,NONE,0,3e94bfef7ebc84f6ae532bba3352c9ad,418
NONE/302091.jpg,302091.jpg,NONE,0,c078ec409be56e300eb10ed68519af08,452
NONE/302095.jpg,302095.jpg,NONE,0,088accc9ed86da9a2e17cad14e5adc7a,455
NONE/302109.jpg,302109.jpg,NONE,0,1e1f17d6638da3a9cbdd2a0f5c256765,459
NONE/302127.jpg,302127.jpg,NONE,0,4d7030306a61e7ad29a6cd01a4fbde58,466
NONE/302135.jpg,302135.jpg,NONE,0,21a5c0928de38e501be07feea00e7a69,469
NONE/302144.jpg,302144.jpg,NONE,0,60a77e69bb8282103ee067ec8afd52e4,474
NONE/302176.jpg,302176.jpg,NONE,0,45629ca17f893866c9fc55aed652adb3,487
NONE/302178.jpg,302178.jpg,NONE,0,1e60549892429f00de80e64fd138cc95,488
NONE/302187.jpg,302187.jpg,NONE,0,b0b99da4b559a6e45888dbb82ba5c75b,492
NONE/302195.jpg,302195.jpg,NONE,0,8e4af15a67f180f05468b3a9a457593d,496
NONE/302200.jpg,302200.jpg,NONE,0,0015d258e828afd627dcf2e275f40bf8,499
NONE/302204.jpg,302204.jpg,NONE,0,492d3fe90b1e90979369af7f7729fd67,502
NONE/302224.jpg,302224.jpg,NONE,0,a894c2d3916f689188ea387fe6c2d8c1,510
NONE/302229.jpg,302229.jpg,NONE,0,970971bd01fdd79b03bb11101deb73a0,512
NONE/302238.jpg,302238.jpg,NONE,0,1b1f0d623ac48c85ef6fd270aad2b515,515
NONE/302254.jpg,302254.jpg,NONE,0,687e618e29cdea0d7d3bd64ebf060d6d,524
NONE/302287.jpg,302287.jpg,NONE,0,e250366b7c545c38d367bf22803ab62c,537
NONE/302294.jpg,302294.jpg,NONE,0,8dab5677c65cab6b675f19bf44d00bd4,540
NONE/302307.jpg,302307.jpg,NONE,0,08bb51c1ec346b4014d945d6a5a72a21,546
NONE/302318.jpg,302318.jpg,NONE,0,2f0e0393b37727aae55d76011df437cb,550
NONE/302337.jpg,302337.jpg,NONE,0,ed4d112a0e3108f6323083ffefb28982,559
NONE/302344.jpg,302344.jpg,NONE,0,0af906482834f7d9476eccb21eee1c02,562
NONE/302346.jpg,302346.jpg,NONE,0,a0733e7898dec5934be31e51cf0b53f7,563
NONE/302356.jpg,302356.jpg,NONE,0,189ba1ccb7038dc466a71c99afa04857,569
NONE/302371.jpg,302371.jpg,NONE,0,82ad85a88f5bd73e78858c939230a011,574
NONE/302381.jpg,302381.jpg,NONE,0,2c47a126f233f4b14f7322d31276588c,580
NONE/302396.jpg,302396.jpg,NONE,0,10807d2c6ab0b7c57f97364b5e4c45b6,586
NONE/302407.jpg,302407.jpg,NONE,0,dacb8c87a8fddebfd70908555969fa79,595
NONE/302415.jpg,302415.jpg,NONE,0,17bdd4f2fd9f1cb5281c56bcfb5977e9,600
NONE/302432.jpg,302432.jpg,NONE,0,6707d710ad1e6c425f055a2bca48504b,606
NONE/302435.jpg,302435.jpg,NONE,0,41fe3b4fe353ec98bc3c6f15d4b1e19d,608
NONE/302455.jpg,302455.jpg,NONE,0,b2b27483a9e17fbe4bd82bc3b8799b5f,616
NONE/302470.jpg,302470.jpg,NONE,0,3dfa525b78ecc69e55a374c01bee2300,623
NONE/302475.jpg,302475.jpg,NONE,0,2defdb62cba0d5cd59eb02d1ebab846e,625
NONE/302477.jpg,302477.jpg,NONE,0,392eb70d67a1b36bec46042b8228f31e,627
NONE/302495.jpg,302495.jpg,NONE,0,21f0a0c875730783bd6cf3b6e665bec2,633
NONE/302502.jpg,302502.jpg,NONE,0,9f6291ce3b72d1fb98a7aa0de324e73c,635
NONE/302513.jpg,302513.jpg,NONE,0,5b049bf45fa079bc6db358dc37ad81a8,639
NONE/302514.jpg,302514.jpg,NONE,0,9edf7fd083bef56acd6dec4383b3bb7b,640
NONE/302515.jpg,302515.jpg,NONE,0,44a0ce09eb7cdd8afe049cc4a7865d00,641
NONE/302531.jpg,302531.jpg,NONE,0,6979e228627adc6765a69e51c7c7bc84,646
NONE/302538.jpg,302538.jpg,NONE,0,9be3df7c86784afa92393313dfcf8fa3,648
NONE/302548.jpg,302548.jpg,NONE,0,e8aa4ab96bd96eec5085a8e65cf9efde,652
NONE/302560.jpg,302560.jpg,NONE,0,d6dce96483f4fe53bbea75312eae526d,656
NONE/302568.jpg,302568.jpg,NONE,0,c8aa039869c071b8bde6e533ceddb62d,658
NONE/302569.jpg,302569.jpg,NONE,0,34d4a534fbe2c462363ccf602a84bb2e,659
NONE/302583.jpg,302583.jpg,NONE,0,e0993fe07c71e8829d48c2dc85eebb31,663
NONE/302608.jpg,302608.jpg,NONE,0,17d3e3eff761e2542a903255c495e571,675
NONE/302623.jpg,302623.jpg,NONE,0,29f47272c16b5f0a92dd88a3b2d59c6a,681
NONE/302632.jpg,302632.jpg,NONE,0,d92b3ab9035da97d964e368e259ffd34,683
NONE/302634.jpg,302634.jpg,NONE,0,78506af3e9c8113ff014ddd6cf45ffc4,684
NONE/302667.jpg,302667.jpg,NONE,0,07860ec4f31d89abad5fd80c700acfb4,698
NONE/302668.jpg,302668.jpg,NONE,0,d66b3c929e730bebd9c82a46a5ee2b37,699
NONE/302676.jpg,302676.jpg,NONE,0,f927f1965f437fee1fa5473e9efa1260,702
NONE/302677.jpg,302677.jpg,NONE,0,35daea5cbd4b64192b56b5a83f563ebb,703
NONE/302693.jpg,302693.jpg,NONE,0,96608099fafe10eaf3ff09f0c696b902,709
NONE/302707.jpg,302707.jpg,NONE,0,daef42f32106d6659c012fb1b06fb136,713
NONE/302712.jpg,302712.jpg,NONE,0,044e9dceb84449781f85b30908490e50,717
NONE/302731.jpg,302731.jpg,NONE,0,e02e1e9dffcf70c7c6e9b53c43abf5f8,728
NONE/302740.jpg,302740.jpg,NONE,0,ec7398d2874ee33503d5106bd5b81094,732
NONE/302756.jpg,302756.jpg,NONE,0,eeed4f7be6553a658a2083fcfa3a235f,740
NONE/302771.jpg,302771.jpg,NONE,0,efed3c4d72b4444b2b31ad839ae848f2,747
NONE/302779.jpg,302779.jpg,NONE,0,1a6f0cad2974128d4eee589b6144d381,750
NONE/302783.jpg,302783.jpg,NONE,0,4648cdf037d19e68904914d507290a6c,753
NONE/302800.jpg,302800.jpg,NONE,0,0ab73edf53ffc11ccd83c1572bcbed95,763
NONE/302804.jpg,302804.jpg,NONE,0,72d8e0cc5d9dcda3fd8af86020ebac80,766
NONE/302810.jpg,302810.jpg,NONE,0,98d370d2551863b50ecafed9a3fe754c,769
NONE/302815.jpg,302815.jpg,NONE,0,c221efadcfdf4ec9d9a14d66ed73df4d,773
NONE/302820.jpg,302820.jpg,NONE,0,e1a142a6acddfacdaa6619cb611973e1,775
NONE/302826.jpg,302826.jpg,NONE,0,dd5f3cfa61482d39649f97b58f0420de,779
NONE/302837.jpg,302837.jpg,NONE,0,0b777e99fd553fa748a1111e40ed3292,784
NONE/302851.jpg,302851.jpg,NONE,0,3d3ea0b76fe5171d5440b267b19666ce,791
NONE/302860.jpg,302860.jpg,NONE,0,596750d71f59d6cbdfaddd5a2a2f9ad8,795
NONE/302862.jpg,302862.jpg,NONE,0,156c5158083f5584d778ae4a00ccf06a,797
NONE/302884.jpg,302884.jpg,NONE,0,5755ec9919a323a2c953218160bfadee,804
NONE/302926.jpg,302926.jpg,NONE,0,731e4afc32b544dffcef80c23b80c43d,815
NONE/302964.jpg,302964.jpg,NONE,0,2d7945d1278427a76f9503dbe391bd34,825
NONE/302971.jpg,302971.jpg,NONE,0,946e03d554280d8e9b3364d1f9c4a203,828
NONE/302982.jpg,302982.jpg,NONE,0,1dce7b2f9c5cb671b4155044f3f459f6,830
NONE/302985.jpg,302985.jpg,NONE,0,669388b20cc3fbabaa20d9419576bb8f,833
NONE/302996.jpg,302996.jpg,NONE,0,491928b0680250e63a142024599170e7,835
NONE/302998.jpg,302998.jpg,NONE,0,12d866031c5454cca843b4466eca2011,837
NONE/303026.jpg,303026.jpg,NONE,0,2b0be6b37ae5a7381015beb6a5b7f781,850
NONE/303028.jpg,303028.jpg,NONE,0,8ddb7955e2979dc94c844325818c1c6f,851
NONE/303031.jpg,303031.jpg,NONE,0,2819bf2fb6af8ae83cf7dbbb61087b1c,853
NONE/303033.jpg,303033.jpg,NONE,0,d5f78ea6f0498d229736675a2d6622b0,854
NONE/303047.jpg,303047.jpg,NONE,0,857b15fa6da5a4566a3ad44c8f04426a,860
NONE/303053.jpg,303053.jpg,NONE,0,5fbbd941c9884ecce7f29c7dca6de420,863
NONE/303070.jpg,303070.jpg,NONE,0,439353616d9c75aab1b43e81fd01441a,870
NONE/303076.jpg,303076.jpg,NONE,0,ea5dd4c65c85f7cf2339dc280068ae48,872
NONE/303083.jpg,303083.jpg,NONE,0,c96758292a67d9ad523f0bc8b60d3bc7,876
NONE/303085.jpg,303085.jpg,NONE,0,053f0afb18927cc8f8a1ca2e4451e008,877
NONE/303086.jpg,303086.jpg,NONE,0,eb187323ad9ce4724d3c35dffeee3487,878
NONE/303098.jpg,303098.jpg,NONE,0,242c33cd774c32eaafadf4407faf86a2,880
NONE/303107.jpg,303107.jpg,NONE,0,aa49bb960867de7c310bc15c506b2aa2,884
NONE/303135.jpg,303135.jpg,NONE,0,fda1d4e2e04cec69f2fa65fc6356344c,894
NONE/303137.jpg,303137.jpg,NONE,0,d8ecd87925fe0d7d7dfd933dbea4cc74,895
NONE/303139.jpg,303139.jpg,NONE,0,83bbc8a63a413fd07dde3d67621d5704,896
NONE/303205.jpg,303205.jpg,NONE,0,a7e9aaf540a5b5387969a55073d46998,920
NONE/303279.jpg,303279.jpg,NONE,0,70dbb70d6877ab943d033f640d0c681b,953
NONE/303281.jpg,303281.jpg,NONE,0,e793aa4097bf31e9a16f50909bbfd34d,954
NONE/303282.jpg,303282.jpg,NONE,0,eedcea914e3beb7f2ede7e70f0d2303a,955
NONE/303283.jpg,303283.jpg,NONE,0,e62baab238f040309219ec368867a701,956
NONE/303284.jpg,303284.jpg,NONE,0,ffd07c64d1ca8c123b7e99d40cd3f5eb,957
NONE/303296.jpg,303296.jpg,NONE,0,b142d606b4384cf1d722b8e214a06ccd,959
NONE/303299.jpg,303299.jpg,NONE,0,8bfbea8d2a5b0c53ded520b7233e3e0a,960
NONE/303324.jpg,303324.jpg,NONE,0,0a14bf6d1dc80afdc68e09664013faa2,971
NONE/303326.jpg,303326.jpg,NONE,0,8925169db76d3e8490d9c50fa6933c7c,972
NONE/303337.jpg,303337.jpg,NONE,0,3c57aee6d9603493a33dca85be2da349,975
NONE/303339.jpg,303339.jpg,NONE,0,ac1ecaaa0c8226b1dfd1c457c6e170fe,977
NONE/303343.jpg,303343.jpg,NONE,0,2de99111b0177beb0daa2c04d14103e3,980
NONE/303366.jpg,303366.jpg,NONE,0,717957ee0223d5d8ad782bef03ec69bf,991
NONE/303367.jpg,303367.jpg,NONE,0,65a12c776aa31b41268f711ade7525f0,992
NONE/303372.jpg,303372.jpg,NONE,0,aae5ef1ca0e9d41a112b5581ce69c7d5,995
NONE/303373.jpg,303373.jpg,NONE,0,36a744c465ead14e847e307ba8b0d41a,996
NONE/303403.jpg,303403.jpg,NONE,0,a34054459d73e404c0669acc4847f04a,1008
NONE/303412.jpg,303412.jpg,NONE,0,086b1af4563767dd65f5502f236d4eef,1012
NONE/303417.jpg,303417.jpg,NONE,0,6ad1050109017597283116b971b2b22c,1015
NONE/303419.jpg,303419.jpg,NONE,0,89aabddb5e9894b61044b9ff599208c6,1016
NONE/303434.jpg,303434.jpg,NONE,0,3332c38831d8d01d7da143cec9842699,1020
NONE/303436.jpg,303436.jpg,NONE,0,bd897bb4b97343d76591bbebb4b6725f,1021
NONE/303440.jpg,303440.jpg,NONE,0,f9d6d46dcc7299ebe7acd7d27e34b3c1,1024
NONE/303450.jpg,303450.jpg,NONE,0,99c1b9dcd647c3565dd7e9b69760a288,1028
NONE/303477.jpg,303477.jpg,NONE,0,46be9ebe0534ccc739c843f3f13d36a5,1038
NONE/303497.jpg,303497.jpg,NONE,0,81839313d3c70611ded590e428bee823,1045
NONE/303512.jpg,303512.jpg,NONE,0,f38c7fcf50c8e4edd50332d3cec83535,1051
NONE/303532.jpg,303532.jpg,NONE,0,343a69ca75f326eca90a830a2dd3314b,1059
NONE/303533.jpg,303533.jpg,NONE,0,b345f6f8a1af93968cce9008f8aecef3,1060
NONE/303537.jpg,303537.jpg,NONE,0,da987277b1f9038360513cf3ca531c0d,1062
NONE/303545.jpg,303545.jpg,NONE,0,c480a0ef075addf78423ca961b60626b,1065
NONE/303547.jpg,303547.jpg,NONE,0,c134ed87bfead56ccd5e2db35d0a9cf6,1066
NONE/303554.jpg,303554.jpg,NONE,0,2a4db7bc517538ff8d0538cc3ecf9e72,1069
NONE/303573.jpg,303573.jpg,NONE,0,b396a1fe7db080daeed71d5b37380188,1076
NONE/303574.jpg,303574.jpg,NONE,0,704b51cecc6d8e465688be5686d53b60,1077
NONE/303580.jpg,303580.jpg,NONE,0,7b83e524d2d358c4b232f5f4b38f5ae8,1081
NONE/303583.jpg,303583.jpg,NONE,0,99359bd8ac18a293cdd94fac2ea41baf,1082
NONE/303613.jpg,303613.jpg,NONE,0,332e22541f2d4abe3ba1fca4f767cc94,1092
NONE/303622.jpg,303622.jpg,NONE,0,7530f81f07c3bab8bb676f4f0439158b,1095
NONE/303669.jpg,303669.jpg,NONE,0,17401eed8da6b3b1ded658146121b759,1115
NONE/303678.jpg,303678.jpg,NONE,0,ab385804611f7b4b0a8e7bec63b99b60,1119
NONE/303681.jpg,303681.jpg,NONE,0,b124fa92e407dcd9ed133e1e9b294a66,1122
NONE/303687.jpg,303687.jpg,NONE,0,0b79b0565f943a8906a3a15d10f5b1f4,1126
NONE/303696.jpg,303696.jpg,NONE,0,9704b64e65a4b14882898d2d0a74bb07,1129
NONE/303699.jpg,303699.jpg,NONE,0,a5b7af3cb38c4142f77b217852a8ba18,1130
NONE/303709.jpg,303709.jpg,NONE,0,3f9c15c959f7eb8c2edf190e92887644,1136
NONE/303722.jpg,303722.jpg,NONE,0,a88959ea1a07432055532848f792cb49,1139
NONE/303730.jpg,303730.jpg,NONE,0,ea662f69ebd1c4e638a76356cb84c669,1141
NONE/303749.jpg,303749.jpg,NONE,0,08ebbed8b540e288205698617b44245c,1147
NONE/303752.jpg,303752.jpg,NONE,0,ac528b8534488ecf21c022735b80571f,1148
NONE/303758.jpg,303758.jpg,NONE,0,9f3bea8b996b972ecb423feb76b79749,1150
NONE/303760.jpg,303760.jpg,NONE,0,2646daf50ab7697290742d400adeef26,1152
NONE/303763.jpg,303763.jpg,NONE,0,fc05771ca755c44e95c25716b2f1e6cf,1153
NONE/303764.jpg,303764.jpg,NONE,0,50d61d8c95d5c7488769afc4850f2a41,1154
NONE/303770.jpg,303770.jpg,NONE,0,51b533b0256f629092e69ed8e180f86c,1155
NONE/303771.jpg,303771.jpg,NONE,0,cdd680ae328fc7c612910d0c3bafabed,1156
NONE/303776.jpg,303776.jpg,NONE,0,a286a0a36700fec3b71a06debc2e7e39,1158
NONE/303789.jpg,303789.jpg,NONE,0,2974776aa3afee603fb94cdcd01301a2,1162
NONE/303794.jpg,303794.jpg,NONE,0,1c2540e6d39f858ac02ae1f0df3e4414,1166
NONE/303795.jpg,303795.jpg,NONE,0,ef608f8fa301f84d79f379b8f1983529,1167
NONE/303827.jpg,303827.jpg,NONE,0,15830c16fce87a449d3daeb8b644f5f9,1180
NONE/303837.jpg,303837.jpg,NONE,0,b24d33c6129a79cf5340d053ece6a0d9,1183
NONE/303840.jpg,303840.jpg,NONE,0,cd14258a75c5ca8876d6de6483ac2c85,1184
NONE/303854.jpg,303854.jpg,NONE,0,6286b62ac159ddfbc5ff6a74eaac8a51,1191
NONE/303867.jpg,303867.jpg,NONE,0,7ec79e25fff374bc65b6f3edec74fb1f,1197
NONE/303880.jpg,303880.jpg,NONE,0,d17f75a41c276f3a806837e1021177c9,1205
NONE/303890.jpg,303890.jpg,NONE,0,ef470267a4296a785a2448f722c8574d,1209
NONE/303893.jpg,303893.jpg,NONE,0,1bb7ccbbc54813d0eeb0c2ac7441d7eb,1212
NONE/303898.jpg,303898.jpg,NONE,0,4a9cb49079715a910cc7d6a2d00a97ad,1215
NONE/303905.jpg,303905.jpg,NONE,0,adc1b00b6a394f7296edee5152b41b11,1217
NONE/303912.jpg,303912.jpg,NONE,0,dfd3189856ef8b27d4fa5a46dc9b6efc,1218
NONE/303914.jpg,303914.jpg,NONE,0,9ab3b799cd03a0b6ddd80fcdb6d72607,1219
NONE/303933.jpg,303933.jpg,NONE,0,a4ebb16fb9d7794bc45865e0b46fbeb7,1227
NONE/303951.jpg,303951.jpg,NONE,0,1f1e7d642660aa4eb3463360a4cb81f3,1238
NONE/303979.jpg,303979.jpg,NONE,0,296748b99d5fe87368e4ee91d1e52e12,1243
NONE/304009.jpg,304009.jpg,NONE,0,f58b87cb553d50df96f5439848748915,1257
NONE/304023.jpg,304023.jpg,NONE,0,e91bd6fcf88a3d9df41a4221f942aba7,1267
NONE/304038.jpg,304038.jpg,NONE,0,7af0bf684f0e291fdf36a222280ccd4d,1275
NONE/304056.jpg,304056.jpg,NONE,0,5fe1f8f7e1c17802bf6d679277a03fc8,1282
NONE/304083.jpg,304083.jpg,NONE,0,77108e2271c2325ad5ba09b46626647e,1291
NONE/304093.jpg,304093.jpg,NONE,0,f6ada26082c2b1f8b548bb6e0057534b,1294
NONE/304102.jpg,304102.jpg,NONE,0,b4918ca4c659c6257aee8f56b822d906,1296
NONE/304106.jpg,304106.jpg,NONE,0,1c2f339f3546df95c2455a5ecaea4fad,1297
NONE/304109.jpg,304109.jpg,NONE,0,40fea05f993127082911d1635179c674,1300
NONE/304139.jpg,304139.jpg,NONE,0,1756826ac29da25875ee0ff3b83ebae6,1306
NONE/304159.jpg,304159.jpg,NONE,0,99c746836fecc90189cad6d777142daf,1312
NONE/304190.jpg,304190.jpg,NONE,0,c551eadbbc89e8654b8149e228b3fc18,1324
NONE/304197.jpg,304197.jpg,NONE,0,c277c8bf4478ca66c21612676cddf8ef,1327
NONE/304241.jpg,304241.jpg,NONE,0,e13c653e24d90d7d6a67b7b5db4e04aa,1346
NONE/304245.jpg,304245.jpg,NONE,0,c1ea587d723eddccab7fd817da8d7a05,1348
NONE/304258.jpg,304258.jpg,NONE,0,d2131f3d873affb5cbd6487e69e0b355,1354
NONE/304276.jpg,304276.jpg,NONE,0,b8b032c964ffbe0af335ed7fbfa86f2a,1363
NONE/304278.jpg,304278.jpg,NONE,0,644f8307093266390690ce84d3876756,1365
NONE/304281.jpg,304281.jpg,NONE,0,4a8b14713dd25b3f52b4857f8ebd6bd2,1367
NONE/304284.jpg,304284.jpg,NONE,0,9552a92fc4e1cdad04d5ef0b26a77bdf,1369
NONE/304322.jpg,304322.jpg,NONE,0,edfbf5747cfdffba23c152dc45fc55b1,1384
NONE/304330.jpg,304330.jpg,NONE,0,4972f3ad36b033b253b7a0846a391ae8,1388
NONE/304333.jpg,304333.jpg,NONE,0,7440954eb132df78a2dafc3be0039975,1390
NONE/304356.jpg,304356.jpg,NONE,0,22f92f5704df1281ce4432238a6fe8f8,1400
NONE/304373.jpg,304373.jpg,NONE,0,c603149053492ba1d95278a4a6650b36,1408
NONE/304374.jpg,304374.jpg,NONE,0,df5a44156c939e4f8c1b9b848d92f67d,1409
NONE/304395.jpg,304395.jpg,NONE,0,1c443c0b415af2e9f0d1d7beab129f60,1415
NONE/304396.jpg,304396.jpg,NONE,0,a77f323254bf3bb3211c02c162ad85d0,1416
NONE/304439.jpg,304439.jpg,NONE,0,ef8d83c289a0057a42b08f9e5ad82c85,1434
NONE/304449.jpg,304449.jpg,NONE,0,74be4b2f801e182655d46f7cfa2fdc96,1438
NONE/304450.jpg,304450.jpg,NONE,0,bfac7fd76fa70bc535d753694261cf1e,1439
NONE/304452.jpg,304452.jpg,NONE,0,8e7fd3e801d04246988986eb6cf5b007,1440
NONE/304457.jpg,304457.jpg,NONE,0,7be09e0117b9f12fbf99c207b5c36a66,1443
NONE/304459.jpg,304459.jpg,NONE,0,bbcb2abac06726e3be9ea41f88d7d184,1445
NONE/304530.jpg,304530.jpg,NONE,0,9a0a52aae54fac6d081a1e56a935d8d2,1470
NONE/304542.jpg,304542.jpg,NONE,0,bf5e3794d06ae26a9ff9a48be03ce77d,1478
NONE/304544.jpg,304544.jpg,NONE,0,f184de55d18ea02ef10421f34219c4f1,1479
NONE/304549.jpg,304549.jpg,NONE,0,3515308bc58a1b265ad70696127f33b4,1482
NONE/304551.jpg,304551.jpg,NONE,0,caa0881fc411bdc3b866ded47eafc56c,1484
NONE/304553.jpg,304553.jpg,NONE,0,1b99d95ec6b7019c9943a3f7d68d2dbf,1485
NONE/304565.jpg,304565.jpg,NONE,0,b7895b4470b0b2525857c6a250b3622d,1492
NONE/304569.jpg,304569.jpg,NONE,0,8858c2ba3084246e2e9fba527cd81475,1495
NONE/304570.jpg,304570.jpg,NONE,0,b5039f47e0028267d3526e630155a4cf,1496
NONE/304587.jpg,304587.jpg,NONE,0,2079a3c411d253344b2e76f33605331f,1500
NONE/304593.jpg,304593.jpg,NONE,0,a61cbca920e10be73ef6dd2f25b98370,1504
NONE/304612.jpg,304612.jpg,NONE,0,e4d008bbedbfc983c0c7b75a1ddd8eca,1515
NONE/304615.jpg,304615.jpg,NONE,0,7cf056c7dd1295c9bad7cea582a43c8c,1516
NONE/304618.jpg,304618.jpg,NONE,0,edbf10b3d94ce4247fba3f8b4a6b3e33,1519
NONE/304619.jpg,304619.jpg,NONE,0,d570cf47797f3a94f8a9be6065e9b527,1520
NONE/304627.jpg,304627.jpg,NONE,0,b19192ac985755f840ca23444a576304,1522
NONE/304643.jpg,304643.jpg,NONE,0,dea4aa58e332ea642e3ac8c7a884e824,1527
NONE/304649.jpg,304649.jpg,NONE,0,9c393d88360bcfb84fd79f8fb2683e92,1528
NONE/304650.jpg,304650.jpg,NONE,0,c741dbd4f7fb515b567b84fb866f554b,1529
NONE/304660.jpg,304660.jpg,NONE,0,1adaadc48946d744a5c476132fb6ac92,1534
NONE/304674.jpg,304674.jpg,NONE,0,d41953c1ac00ac2412ce036086a984f1,1540
NONE/304682.jpg,304682.jpg,NONE,0,7ef6efacfef2ec07695800877361a6d9,1542
NONE/304693.jpg,304693.jpg,NONE,0,29d420bd9a2ae329cd1829b5d2e95602,1548
NONE/304743.jpg,304743.jpg,NONE,0,15c0084e6903edf3065af31663b0ab76,1568
NONE/304751.jpg,304751.jpg,NONE,0,ac661ae54db5790f324832f63fba21e3,1570
NONE/304753.jpg,304753.jpg,NONE,0,61fce8ac92e583fcf9bd2cffb23a6934,1571
NONE/304783.jpg,304783.jpg,NONE,0,c4f1c30e35f4355690b126c06cb9a58b,1579
NONE/304808.jpg,304808.jpg,NONE,0,080dfc6310eac7651e42dcbb21ef66b4,1585
NONE/304819.jpg,304819.jpg,NONE,0,c8d35e316cfa819a303e3026dab50510,1589
NONE/304831.jpg,304831.jpg,NONE,0,89af8495fd34ec162f5874495670ebda,1595
NONE/304836.jpg,304836.jpg,NONE,0,bcad6597ec5616b7eed6875a0b7ee655,1597
NONE/304843.jpg,304843.jpg,NONE,0,ca8ee8db9dfb4fab78ed22139063ad20,1603
NONE/304894.jpg,304894.jpg,NONE,0,d487cdec3e9676ef64b9a48873493b37,1625
NONE/304904.jpg,304904.jpg,NONE,0,bb634b61fdfed92274736533e9c2a247,1630
NONE/304905.jpg,304905.jpg,NONE,0,9e2ac2ec466aeab55150eca4858b029b,1631
NONE/304914.jpg,304914.jpg,NONE,0,70bf00a6229ea569e9834efa175b116b,1634
NONE/304920.jpg,304920.jpg,NONE,0,752d332a9608773034962ed39e3ef33e,1635
NONE/304922.jpg,304922.jpg,NONE,0,782b7da071342f757165ecd1505d2dd8,1636
NONE/304924.jpg,304924.jpg,NONE,0,6c498ae40052e4e673ab15719bc46930,1638
NONE/304941.jpg,304941.jpg,NONE,0,5f633d228d1670e19071bfdda87ec789,1643
NONE/304954.jpg,304954.jpg,NONE,0,ce4ab9113dbb89a73f13aac7ac44f8db,1648
NONE/304956.jpg,304956.jpg,NONE,0,b0df160d0c0e656957a70a0cd5b696e1,1649
NONE/304974.jpg,304974.jpg,NONE,0,c4871ca32feb85b49c31af5904e80af0,1661
NONE/304980.jpg,304980.jpg,NONE,0,9ca6bf00fb77916209cfb488c0aacd05,1663
NONE/304981.jpg,304981.jpg,NONE,0,d50d47756632995fe001c6746476ac11,1664
NONE/305003.jpg,305003.jpg,NONE,0,08ce72be71f07d68decd07b04b7d7d1d,1676
NONE/305007.jpg,305007.jpg,NONE,0,f5c2c86bcf90c69e41530b2b8485f107,1678
NONE/305010.jpg,305010.jpg,NONE,0,cd6a9cadc0791eeba753ce2b38c2db88,1679
NONE/305012.jpg,305012.jpg,NONE,0,1a33a3940da9912a102c83dcf084800f,1680
NONE/305016.jpg,305016.jpg,NONE,0,aec50ff73eaf70de0c5188cc99c71113,1682
NONE/305020.jpg,305020.jpg,NONE,0,32d1d572d6baa12ff379bd712e7547ce,1683
NONE/305025.jpg,305025.jpg,NONE,0,c5acfd946d80598330a4b03db8e43ce8,1685
NONE/305029.jpg,305029.jpg,NONE,0,ebf6a4f01bc0349718ba14e8a2cbcf31,1686
NONE/305031.jpg,305031.jpg,NONE,0,f26e4d9f8fbcb5e81535783dd1eacb95,1687
NONE/305040.jpg,305040.jpg,NONE,0,ea110b10a5360ceaf055d7a45e8d4147,1693
NONE/305044.jpg,305044.jpg,NONE,0,935e111c8a28d4a987820662ba1ca9d6,1695
NONE/305048.jpg,305048.jpg,NONE,0,1751cdb06ed54fef5d2a5001c3c12bd1,1696
NONE/305050.jpg,305050.jpg,NONE,0,e5f734ab53dcfc0f0c68c5ff6c814ec3,1698
NONE/305063.jpg,305063.jpg,NONE,0,c0bf94aff37cdb861b8d9ae779595690,1704
NONE/305066.jpg,305066.jpg,NONE,0,d2d2e090ba4e8253f1051c29793e74de,1705
NONE/305070.jpg,305070.jpg,NONE,0,a3ac937504f67d2af060a2d70f8b0fce,1707
NONE/305087.jpg,305087.jpg,NONE,0,8e5110b677720fae0ea0ee28e247ec5b,1715
NONE/305103.jpg,305103.jpg,NONE,0,29c2f705473b5b00e543a21809b10633,1722
NONE/305120.jpg,305120.jpg,NONE,0,8cfbc07dab816a323d977d197c86648b,1733
NONE/305124.jpg,305124.jpg,NONE,0,240e71fe192f5a31ea5c01c6072a535c,1734
NONE/305170.jpg,305170.jpg,NONE,0,7f352dd79cd533a6157b5e9a79ab3322,1751
NONE/305182.jpg,305182.jpg,NONE,0,84e4cead2859472c44bc02b2ba53a52b,1757
NONE/305183.jpg,305183.jpg,NONE,0,3f3200e75eae3706890ff14b36efc4f8,1758
NONE/305184.jpg,305184.jpg,NONE,0,bb398f52d4315c13cb4699f00f06ac21,1759
NONE/305203.jpg,305203.jpg,NONE,0,728c149fecab95a538697eb4cf34460c,1768
NONE/305212.jpg,305212.jpg,NONE,0,0f9db390c52c37e2911e0edda00f8377,1773
NONE/305234.jpg,305234.jpg,NONE,0,bf474dc0407ea996bba13162608bfc0c,1781
NONE/305236.jpg,305236.jpg,NONE,0,557ad9c66c4a2ac1fdb066124e090840,1782
NONE/305242.jpg,305242.jpg,NONE,0,2e72926ca44c5cdb3a9bdcca1483ea0d,1785
NONE/305247.jpg,305247.jpg,NONE,0,511ec800890b81be6a592cdf322481aa,1787
NONE/305261.jpg,305261.jpg,NONE,0,39bffbc32f347ee4773262e1ec4a55b7,1794
NONE/305272.jpg,305272.jpg,NONE,0,d6e026d78c221f67776d96595aea6aac,1796
NONE/305278.jpg,305278.jpg,NONE,0,b1eb60af86bcaae35a9139bf006ab664,1799
NONE/305282.jpg,305282.jpg,NONE,0,81f97c5185b4a22a81ebe4af8f46cd15,1802
NONE/305288.jpg,305288.jpg,NONE,0,98637c44e3e30122db455ac83d023bdd,1806
NONE/305293.jpg,305293.jpg,NONE,0,708bd88b55da3a74f363aaeee555c920,1810
NONE/305312.jpg,305312.jpg,NONE,0,5e0d01bf7bf5224a16ca98505d595967,1816
NONE/305317.jpg,305317.jpg,NONE,0,8ba8057fcf751403849b464773fbbfd3,1819
NONE/305319.jpg,305319.jpg,NONE,0,a8861f5d53992cbdecaece9f53a28bae,1821
NONE/305326.jpg,305326.jpg,NONE,0,9cde1129e421f32e7733be4386d07ccb,1823
NONE/305331.jpg,305331.jpg,NONE,0,b459bcc7b4c14d0b08a5f0439d4c16b6,1827
NONE/305368.jpg,305368.jpg,NONE,0,8e16c6c897154ffb3043034e574bbc5a,1841
NONE/305371.jpg,305371.jpg,NONE,0,5bb6484cee784ed6a9b80de5c5831fa3,1843
NONE/305388.jpg,305388.jpg,NONE,0,275e331e8cb7004061a718cd25e9f40b,1853
NONE/305393.jpg,305393.jpg,NONE,0,227d050b863b20e1acab111612cb6a7b,1856
NONE/305402.jpg,305402.jpg,NONE,0,f50e06a23a51843739c476631d053cab,1863
NONE/305439.jpg,305439.jpg,NONE,0,14ed99596dee24915ea4585cd88e6f98,1881
NONE/305444.jpg,305444.jpg,NONE,0,3720d35e2f12bc073ee2107904d30c67,1884
NONE/305455.jpg,305455.jpg,NONE,0,c04ffcfbd1898951503582c486578f91,1890
NONE/305462.jpg,305462.jpg,NONE,0,d78a816905c2d0c3af55b0fc2fd7b627,1892
NONE/305475.jpg,305475.jpg,NONE,0,af684e45a4864a0456221cd94c9cf5e3,1895
NONE/305482.jpg,305482.jpg,NONE,0,9b8f080b41b2e7e5f4847049c5a3ca72,1899
NONE/305493.jpg,305493.jpg,NONE,0,ee78f296da8d3d426f7e80723d225a7d,1903
NONE/305501.jpg,305501.jpg,NONE,0,d57c14ed2e789e04b8d34d02df36a439,1906
NONE/305526.jpg,305526.jpg,NONE,0,5964ed9e7de16ddace9116d305f86fe1,1919
NONE/305531.jpg,305531.jpg,NONE,0,939c95410f673be2a7b1d959840152af,1923
NONE/305565.jpg,305565.jpg,NONE,0,03192933e219dc6a4e67b2da50189cb6,1930
NONE/305572.jpg,305572.jpg,NONE,0,e03c2642587ce7b102520b1330c863ff,1931
NONE/305631.jpg,305631.jpg,NONE,0,c0469f67465c6667ff3bc8d34fecbbdf,1955
NONE/305646.jpg,305646.jpg,NONE,0,4e8e66ce50f756fa3c15572cf510c7ee,1958
NONE/305647.jpg,305647.jpg,NONE,0,d3e9de1de055ec5595cda68a8af6df79,1959
NONE/305670.jpg,305670.jpg,NONE,0,4a132c7446e2b6fd6146552e6e144287,1969
NONE/305679.jpg,305679.jpg,NONE,0,f74e746f3f056da6ef386dc0f8d4c2f3,1974
NONE/305691.jpg,305691.jpg,NONE,0,5eb5504587c95273c571ba08f3705f69,1981
NONE/305692.jpg,305692.jpg,NONE,0,1d0a7aec5bdf6ed8b82c97aba1486039,1982
NONE/305704.jpg,305704.jpg,NONE,0,f6e85275cf654ff097569318dabcc9af,1986
NONE/305727.jpg,305727.jpg,NONE,0,f549f6ad54b991933f30bedfa2451b07,1996
NONE/305732.jpg,305732.jpg,NONE,0,b2ebf14899b085a3772eddb6e8304d37,1998
NONE/305749.jpg,305749.jpg,NONE,0,025be009953cae37e09ff76f16f4c991,2006
NONE/305786.jpg,305786.jpg,NONE,0,1d1dade0929a2b78024b5330d6fd2c89,2027
NONE/305812.jpg,305812.jpg,NONE,0,a580f38e4c2a3244314f83469d1b0486,2038
NONE/305820.jpg,305820.jpg,NONE,0,bc0e5f2ef9eac2f253bd2c6cbdf370b8,2041
NONE/305833.jpg,305833.jpg,NONE,0,be2dad71045ea17aa879c09e95f0045b,2047
NONE/305838.jpg,305838.jpg,NONE,0,ddc8f8924cac61f4a6ef0d3caf6e3e3d,2049
NONE/305857.jpg,305857.jpg,NONE,0,43fc9925f6cfbb4c29544e12d60bb7ca,2056
NONE/305867.jpg,305867.jpg,NONE,0,6f40c8f1206c16c9772c8ac5a5840b73,2061
NONE/305869.jpg,305869.jpg,NONE,0,d255400a214cfe8774ca3e9a8cdfb70c,2062
NONE/305876.jpg,305876.jpg,NONE,0,f334dc2ffe2ecdb8752ab4eca796c395,2065
NONE/305881.jpg,305881.jpg,NONE,0,3f3ee92f55cc11f5f9f6f2cb68aeec62,2068
NONE/305889.jpg,305889.jpg,NONE,0,84b5842c6a961b3abe9cec33d74a2a88,2070
NONE/305903.jpg,305903.jpg,NONE,0,7c52ff1d7d91fd7a0182f8735596a433,2074
NONE/305914.jpg,305914.jpg,NONE,0,0f48ac630a01b7f9d5245f606cc1feb4,2082
NONE/305919.jpg,305919.jpg,NONE,0,fb5a320554bd63ef856f5c9471f71474,2085
NONE/305924.jpg,305924.jpg,NONE,0,7e581e6fa9951a7a92604121d6ffbe7f,2088
NONE/305948.jpg,305948.jpg,NONE,0,b5e25eb6a559c79860100677f79b8e70,2101
NONE/305949.jpg,305949.jpg,NONE,0,3c647fe2793c8cb054d103a596422da5,2102
NONE/305962.jpg,305962.jpg,NONE,0,b2471c71cb264b2b5d51e7d552301be2,2106
NONE/305968.jpg,305968.jpg,NONE,0,c70054e79f072b5735fc67ad7884ddb6,2109
NONE/306010.jpg,306010.jpg,NONE,0,b5bdf61ab599ff55664d5dfe7a9d9084,2123
NONE/306042.jpg,306042.jpg,NONE,0,f508742eb11e66122b0c8996aca7fd3e,2135
NONE/306053.jpg,306053.jpg,NONE,0,7170b6403494ba2879b02046dc4e1809,2139
NONE/306054.jpg,306054.jpg,NONE,0,523f17c78ee8bc9bdca425f4648c2209,2140
NONE/306057.jpg,306057.jpg,NONE,0,e7987ec681aeef7345763f18f63ee1fa,2141
NONE/306065.jpg,306065.jpg,NONE,0,9590a6da6c7879a8165127d6624d7e65,2146
NONE/306068.jpg,306068.jpg,NONE,0,b08a4935ba9b92b4fc4bf55c350e8251,2147
NONE/306094.jpg,306094.jpg,NONE,0,58d0db1fe9225be424bb61e725302580,2160
NONE/306098.jpg,306098.jpg,NONE,0,488bdad6ddc0a85386fb0366513e60ee,2163
NONE/306119.jpg,306119.jpg,NONE,0,b5bf31108ed6eadcec42ade946cf0ec3,2169
NONE/306121.jpg,306121.jpg,NONE,0,66caa649e691833e1ad872ecabde9410,2170
NONE/306129.jpg,306129.jpg,NONE,0,fffaccafddf2064f7628bebe832f2ad4,2172
NONE/306156.jpg,306156.jpg,NONE,0,eef7d33a768a8c8511d3dc8d89ee7ae8,2182
NONE/306166.jpg,306166.jpg,NONE,0,2addf6f4b7c834a946f1492bf96c2157,2186
NONE/306167.jpg,306167.jpg,NONE,0,0a57f61c30a077c2b66a73d7167d9cd1,2187
NONE/306174.jpg,306174.jpg,NONE,0,032b770551bd4d1ca9bc27d674456bc9,2190
NONE/306176.jpg,306176.jpg,NONE,0,8e91d0594a3607a8167b8dd02513c868,2191
NONE/306191.jpg,306191.jpg,NONE,0,200e488dcb8fc891353a073e7d3b2bb1,2197
NONE/306193.jpg,306193.jpg,NONE,0,b3ebc6442e7c08708c9f215e51712d1e,2198
NONE/306194.jpg,306194.jpg,NONE,0,c2e7b46936201f38aa0757ce5769a032,2199
NONE/306197.jpg,306197.jpg,NONE,0,57a4f0da115028077eb925b5d1d45d1c,2200
NONE/306217.jpg,306217.jpg,NONE,0,4ddc7438e45d50e1b323c7663b5aa225,2205
NONE/306239.jpg,306239.jpg,NONE,0,1b829c43bccef3f5660caf705fe37fcb,2214
NONE/306260.jpg,306260.jpg,NONE,0,63b678a851b7730a88f3ba30539ae092,2220
NONE/306325.jpg,306325.jpg,NONE,0,27031f453e514791c154287973512db6,2235
NONE/306329.jpg,306329.jpg,NONE,0,3612d8778d967d2b276fc70545d33946,2237
NONE/306358.jpg,306358.jpg,NONE,0,56688fcf8ed5cd78bcbf93799d717c3f,2253
NONE/306362.jpg,306362.jpg,NONE,0,7867f3fb2d9693ff46c021a79b5ec152,2255
NONE/306384.jpg,306384.jpg,NONE,0,42d21c930d9f3cf30db35b2fb4fbf972,2264
NONE/306390.jpg,306390.jpg,NONE,0,dcdf3a2ce1abe4cb43ef8fc8260a0466,2267
NONE/306401.jpg,306401.jpg,NONE,0,3b0ac459c80e61ae2fbe598fd180fb09,2272
NONE/306413.jpg,306413.jpg,NONE,0,d6b188150633e2bba58995c561622a04,2277
NONE/306415.jpg,306415.jpg,NONE,0,e00a4636f64ebcd84eb74e4b450979b2,2278
NONE/306436.jpg,306436.jpg,NONE,0,ac61a5b80f770bf377cb89231b079be2,2288
NONE/306444.jpg,306444.jpg,NONE,0,5fd1eb5a05f6317862fdb271de0be125,2294
NONE/306460.jpg,306460.jpg,NONE,0,96f88e6d4585021996521dc2dbf05f44,2300
NONE/306464.jpg,306464.jpg,NONE,0,70d101ceca8c99ac12cc0f60d9d1cd8f,2301
NONE/306473.jpg,306473.jpg,NONE,0,8e24a0667bcdaff0973929a3dc390a09,2306
NONE/306480.jpg,306480.jpg,NONE,0,be6de34d186d5ce4edd9252bc5a75ee5,2309
NONE/306501.jpg,306501.jpg,NONE,0,0ccc7dd232af38fcd3d6d1c24ffdee6a,2316
NONE/306502.jpg,306502.jpg,NONE,0,5d2c9f35eaee15e5fcb948cfb9afb58b,2317
NONE/306516.jpg,306516.jpg,NONE,0,05a8aaf5da731f929f67790cfb4eae22,2324
NONE/306521.jpg,306521.jpg,NONE,0,8d9cdcef70d73bdf27b12579d4a623df,2327
NONE/306534.jpg,306534.jpg,NONE,0,2bf97182bf7702631de4b9d30fb48f50,2330
NONE/306548.jpg,306548.jpg,NONE,0,e97238290655b8eb744a27d660f72286,2336
NONE/306558.jpg,306558.jpg,NONE,0,1da61364a5666912cca96cf8c248b62a,2342
NONE/306559.jpg,306559.jpg,NONE,0,8e79a3901c177e88eae80aed4ae44ce8,2343
NONE/306578.jpg,306578.jpg,NONE,0,adb989526415eaf8f6f5e25f5245df3c,2354
NONE/306592.jpg,306592.jpg,NONE,0,2be1fcfeeb86d83ac316874d08d3ae7e,2361
NONE/306595.jpg,306595.jpg,NONE,0,b7cd20b38a1df20a4cf47391db15ab2c,2363
NONE/306596.jpg,306596.jpg,NONE,0,17c0d0e402502b9af999f8c3c358cfa5,2364
NONE/306603.jpg,306603.jpg,NONE,0,4516ff66910ccf1531ef72fbc035dcb1,2367
NONE/306643.jpg,306643.jpg,NONE,0,954866fb22e32103819eec0edac1d83f,2386
NONE/306645.jpg,306645.jpg,NONE,0,8fcfd6bfad3bf14e6ca5be5678917823,2387
NONE/306646.jpg,306646.jpg,NONE,0,0917acb84f7d1aca4e50234c3ee8ca69,2388
NONE/306656.jpg,306656.jpg,NONE,0,d7adda5c956b86d9b0f25c56bc0adaf8,2391
NONE/306661.jpg,306661.jpg,NONE,0,f1d7f10591f6b9c42bf307e9505f3b93,2394
NONE/306687.jpg,306687.jpg,NONE,0,8af6de452cde31f40f94258b9c5fa4f6,2405
NONE/306703.jpg,306703.jpg,NONE,0,acf4df325b21434cdb59460f48143a94,2411
NONE/306709.jpg,306709.jpg,NONE,0,67f4459e275ae30aaefe382c725d5264,2413
NONE/306710.jpg,306710.jpg,NONE,0,dd6ab4ca2b25d4ce4e8ac76c6821ebe9,2414
NONE/306736.jpg,306736.jpg,NONE,0,c6980533248ac9346e8b16db1e41c4c7,2422
NONE/306738.jpg,306738.jpg,NONE,0,2d443db5b7702f7f1d375569dfd59142,2424
NONE/306742.jpg,306742.jpg,NONE,0,952cced4ce4034df2cd97245e2500c09,2428
NONE/306743.jpg,306743.jpg,NONE,0,f35b485186dd103dc93f2b85c09be57f,2429
NONE/306752.jpg,306752.jpg,NONE,0,a4dfd6c6fb1402deb11abca1bd2c19ef,2433
NONE/306755.jpg,306755.jpg,NONE,0,6e971d4cbb8e440a5f44cfd0a66ffa19,2435
NONE/306758.jpg,306758.jpg,NONE,0,7cfa5b12a33555e06c60b67fb8b529db,2436
NONE/306776.jpg,306776.jpg,NONE,0,61d9313a50c1d8e166cb0fb1ff662eed,2441
NONE/306788.jpg,306788.jpg,NONE,0,7b3462fa96c9b39a204a50087bbc05f1,2447
NONE/306789.jpg,306789.jpg,NONE,0,6cc5ab28523c72cd07208e4a37929e47,2448
NONE/306790.jpg,306790.jpg,NONE,0,57bfa790b0b5e659df2088c1de24056d,2449
NONE/306803.jpg,306803.jpg,NONE,0,0546132e33640c1b8b5dca8dfa1953ba,2451
NONE/306812.jpg,306812.jpg,NONE,0,cb0c7e563baa7219cacb0f10699de240,2457
NONE/306830.jpg,306830.jpg,NONE,0,cb0e31701cab5ad9098f92582cbd2c37,2466
NONE/306855.jpg,306855.jpg,NONE,0,fbf299a60371cb5ac79bac9effa105b5,2479
NONE/306860.jpg,306860.jpg,NONE,0,a36da148151b7d5874f1ff5209437eff,2481
NONE/306864.jpg,306864.jpg,NONE,0,e6c330c91412665b18d7513e27ca6e4a,2483
NONE/306868.jpg,306868.jpg,NONE,0,a8501784715449524121771c68a066b4,2485
NONE/306869.jpg,306869.jpg,NONE,0,cf0f9137b0db9eea76bf6d40e3ab446a,2486
NONE/306878.jpg,306878.jpg,NONE,0,1c7fad6e446fb95c492746c15f8ce3ff,2489
NONE/306891.jpg,306891.jpg,NONE,0,a2389e0a2f807a0d349c7e2681a2ec47,2495
NONE/306892.jpg,306892.jpg,NONE,0,c9e0e1d7a60468c7aa14fbaba61e7239,2496
NONE/306895.jpg,306895.jpg,NONE,0,2ccd1d0f3f058c02220e5a8b6714b204,2499
NONE/306900.jpg,306900.jpg,NONE,0,46d24e6e5c627b89671c543cf42733f1,2502
NONE/306904.jpg,306904.jpg,NONE,0,a61693ee9b0868b17c4e9e2742c7dc75,2504
NONE/306907.jpg,306907.jpg,NONE,0,8de13d1f3c93909d8914f405b041f868,2506
NONE/306947.jpg,306947.jpg,NONE,0,2bfa37ec3b342f2a26064b4b7a4d8e09,2520
NONE/306969.jpg,306969.jpg,NONE,0,706ef378f8d16b88e00c2a7e810dd0d3,2528
NONE/306976.jpg,306976.jpg,NONE,0,cdc7e182d68ecd99b8fde1184951beae,2530
NONE/306978.jpg,306978.jpg,NONE,0,f1b072a02f00bf7d3c5c4d62bf7e083f,2531
NONE/306986.jpg,306986.jpg,NONE,0,b676ada2aeff1a9f1731aa3dafcc06f4,2534
NONE/306988.jpg,306988.jpg,NONE,0,bb4b2c165ce91c07784624812dc74429,2535
NONE/306991.jpg,306991.jpg,NONE,0,ca01952c07e090a3174b9c8cae39e148,2537
NONE/306999.jpg,306999.jpg,NONE,0,b3ca05f40ebc0a7b43ff9a08889b5ddb,2541
NONE/307007.jpg,307007.jpg,NONE,0,5546431f99a8235204b8bd14e59be7b7,2544
INFECTION/301002.jpg,301002.jpg,INFECTION,1,812f264e4c18fd3607b4817a7de47a05,2552
INFECTION/301026.jpg,301026.jpg,INFECTION,1,9f14e89a1ca8c5d6b3a528a583451407,2559
INFECTION/301058.jpg,301058.jpg,INFECTION,1,477f98bcb38ceb68beba895c428277be,2570
INFECTION/301100.jpg,301100.jpg,INFECTION,1,c1862b7ead0bf74ee41c163fffcd6d8d,2591
INFECTION/301101.jpg,301101.jpg,INFECTION,1,d1a7d81b781053ae5c770a92bbebbce6,2592
INFECTION/301144.jpg,301144.jpg,INFECTION,1,7c24e31e4f7d78888b9a06fe2419e550,2611
INFECTION/301159.jpg,301159.jpg,INFECTION,1,5a574fb62263ed87b3edec181a464876,2619
INFECTION/301180.jpg,301180.jpg,INFECTION,1,5bac6b5bb1004c019a2138f4e7b72c7d,2626
INFECTION/301195.jpg,301195.jpg,INFECTION,1,a2f7be506d796985a93f2b80e0c5de0d,2634
INFECTION/301227.jpg,301227.jpg,INFECTION,1,4178122aafef0ec231f8263cbe863623,2649
INFECTION/301270.jpg,301270.jpg,INFECTION,1,deb97157f21532487445ca1ec28801f6,2675
INFECTION/301295.jpg,301295.jpg,INFECTION,1,4ca21ee135a808f17d3f6378782fb2fb,2683
INFECTION/301310.jpg,301310.jpg,INFECTION,1,eb9242a9cc86c6073f4ba8b7ea5d9edb,2689
INFECTION/301316.jpg,301316.jpg,INFECTION,1,5c64d7aa13f74da80ed8f7d545496895,2693
INFECTION/301341.jpg,301341.jpg,INFECTION,1,8dbf00c41ec830328ab7f49344790fbb,2705
INFECTION/301347.jpg,301347.jpg,INFECTION,1,ba5775626a99101caedfe874b5f9b6a4,2707
INFECTION/301362.jpg,301362.jpg,INFECTION,1,a2746d4f4ce807b136779848e52f6712,2712
INFECTION/301363.jpg,301363.jpg,INFECTION,1,873cd67dca29bb25603b7543c8a9907d,2713
INFECTION/301364.jpg,301364.jpg,INFECTION,1,57216b555af74e4b63c195e6acdc6edd,2714
INFECTION/301369.jpg,301369.jpg,INFECTION,1,94bab0f2e911fac04558e80a5dda6840,2716
INFECTION/301389.jpg,301389.jpg,INFECTION,1,bd94bbde787cb6dd47f871eb2a20ff10,2725
INFECTION/301393.jpg,301393.jpg,INFECTION,1,0bd7ddedbb16ffae9cf9ef8f3ca8a150,2728
INFECTION/301399.jpg,301399.jpg,INFECTION,1,ba5f4e9e47d7002c0175bed657e17080,2730
INFECTION/301409.jpg,301409.jpg,INFECTION,1,5e7533a8211a763fdc603d31b9ff6675,2734
INFECTION/301420.jpg,301420.jpg,INFECTION,1,4e315bcaeff2fd969b6dfcbf7cc80943,2739
INFECTION/301424.jpg,301424.jpg,INFECTION,1,7dc51dc685bb230bad5f896950906213,2741
INFECTION/301436.jpg,301436.jpg,INFECTION,1,6da18a9696ca7967624019bae3d749a5,2745
INFECTION/301445.jpg,301445.jpg,INFECTION,1,83660cb1d4434884c97e8a49c5874160,2749
INFECTION/301457.jpg,301457.jpg,INFECTION,1,c1d31bc2a6324ebebd82a2cb8a4d4aaa,2756
INFECTION/301512.jpg,301512.jpg,INFECTION,1,7a2a6a2ae991a31401928dd8ef35e54f,2778
INFECTION/301564.jpg,301564.jpg,INFECTION,1,02c7b00c8e37529c681ec869bc657bbe,2798
INFECTION/301591.jpg,301591.jpg,INFECTION,1,f8568cfb07fd78e7b30baee8544a9206,2810
INFECTION/301597.jpg,301597.jpg,INFECTION,1,8a2856ee769b5a7a70d833cff287a37e,2813
INFECTION/301603.jpg,301603.jpg,INFECTION,1,a95e70892147805e4d2c9b7bc266a415,2816
INFECTION/301610.jpg,301610.jpg,INFECTION,1,6834f7243e6dab89716e0225569aa25c,2820
INFECTION/301612.jpg,301612.jpg,INFECTION,1,3fe9a0114c4fc3806a9a1c308cea1b57,2821
INFECTION/301638.jpg,301638.jpg,INFECTION,1,7c1acd240ab2b953509cae2d04209b49,2832
INFECTION/301642.jpg,301642.jpg,INFECTION,1,3a47ef934fdf0d82f870e9e9c29d5684,2834
INFECTION/301647.jpg,301647.jpg,INFECTION,1,0f22c9238608ee49c510545acb4abf23,2837
INFECTION/301652.jpg,301652.jpg,INFECTION,1,049f6529a5eea7039b9d1254ac56d1d3,2840
INFECTION/301688.jpg,301688.jpg,INFECTION,1,efb33a6ec60ce1c01949c46fc43bed21,2858
INFECTION/301715.jpg,301715.jpg,INFECTION,1,913ea123d9258461e31e2f67c6b5f02b,2869
INFECTION/301747.jpg,301747.jpg,INFECTION,1,d5a565580d7a42ecca85c04868a3f8de,2883
INFECTION/301755.jpg,301755.jpg,INFECTION,1,406335c814a8c080e0d004b7c9d6bd49,2887
INFECTION/301761.jpg,301761.jpg,INFECTION,1,ff12d7c855b4d2c7994e7cc23fa4095b,2889
INFECTION/301767.jpg,301767.jpg,INFECTION,1,a346a8e79c903608dcdb82e8452eb58e,2892
INFECTION/301770.jpg,301770.jpg,INFECTION,1,3dc6d60b41467645b781bc1db840b4af,2894
INFECTION/301771.jpg,301771.jpg,INFECTION,1,b1156ffe0ceba846cb6a93f2b4b17eb6,2895
INFECTION/301808.jpg,301808.jpg,INFECTION,1,102b3adbae0358feb560e9ad01a89ebd,2913
INFECTION/301838.jpg,301838.jpg,INFECTION,1,5f215567acf707012ae113378c1e7134,2922
INFECTION/301872.jpg,301872.jpg,INFECTION,1,6c4203983d17e04286d3368dbb58c891,2935
INFECTION/301893.jpg,301893.jpg,INFECTION,1,7b940028e81367b1ec7d78f5f26dc0e6,2944
INFECTION/301902.jpg,301902.jpg,INFECTION,1,05f164c11c09e4053467be32e07b9219,2948
INFECTION/301909.jpg,301909.jpg,INFECTION,1,42dcb6a226dbafe9e65d18a0808c021a,2949
INFECTION/301910.jpg,301910.jpg,INFECTION,1,e2295b8fe4b7b79cdcb085aa0c3928e4,2950
INFECTION/301922.jpg,301922.jpg,INFECTION,1,dd16218ac98535029e80cb78e3d40aa0,2955
INFECTION/301927.jpg,301927.jpg,INFECTION,1,fd40662531a0abaccc9f0af40f9ab7c5,2957
INFECTION/301960.jpg,301960.jpg,INFECTION,1,442ba8bed253491b5fe40df20d29ed8f,2971
INFECTION/301983.jpg,301983.jpg,INFECTION,1,fb5d2dd0a3321f8b19afbd42d5efa8f4,2975
INFECTION/301987.jpg,301987.jpg,INFECTION,1,299f90a914932c2386b7806c7d13577a,2976
INFECTION/301999.jpg,301999.jpg,INFECTION,1,021133c10297f073bfaba99672bf6c2c,2983
INFECTION/302013.jpg,302013.jpg,INFECTION,1,0163bdfbae5afae30b949970f734edce,2986
INFECTION/302031.jpg,302031.jpg,INFECTION,1,aab7f7d6237b7de9d7987019dd01fd6d,2992
INFECTION/302032.jpg,302032.jpg,INFECTION,1,eddc05f9de3033f1aeeaf3cb0f8e0648,2993
INFECTION/302043.jpg,302043.jpg,INFECTION,1,49747034be38fc3c115c1e81aae862bc,2998
INFECTION/302063.jpg,302063.jpg,INFECTION,1,c940e43758141d3fb24746f5dcba08af,3007
INFECTION/302067.jpg,302067.jpg,INFECTION,1,26a256212464629e4ca7f07400e8be38,3011
INFECTION/302087.jpg,302087.jpg,INFECTION,1,7c88417e42709f43832e50b646322438,3019
INFECTION/302110.jpg,302110.jpg,INFECTION,1,73f276d25e7ac60ad0d65cae25538a97,3027
INFECTION/302120.jpg,302120.jpg,INFECTION,1,f7a43f9c6f06023d43fad4b98494d57d,3033
INFECTION/302124.jpg,302124.jpg,INFECTION,1,131936b5f02d01697c4f8afd96f50c00,3034
INFECTION/302130.jpg,302130.jpg,INFECTION,1,78f061395f7cb77d9958fec78e5c52ff,3036
INFECTION/302156.jpg,302156.jpg,INFECTION,1,966a3bdefa341c8190342f1a117249e8,3045
INFECTION/302170.jpg,302170.jpg,INFECTION,1,d13ac0234892d9e26aabff7ea613d290,3051
INFECTION/302172.jpg,302172.jpg,INFECTION,1,08775a536cfcbb4a09abd5623c6e3315,3052
INFECTION/302183.jpg,302183.jpg,INFECTION,1,f853343b3866e298628c5479f80a04be,3056
INFECTION/302219.jpg,302219.jpg,INFECTION,1,7f5111fe67d504ad6786f584d395525e,3071
INFECTION/302230.jpg,302230.jpg,INFECTION,1,3238e4b8090869736501dbc2f3afd6d2,3077
INFECTION/302232.jpg,302232.jpg,INFECTION,1,18ba36a7ba5d2c551fa1c41cfefde7a6,3078
INFECTION/302235.jpg,302235.jpg,INFECTION,1,beb036740c8ee5eecf196d5c042347f3,3080
INFECTION/302245.jpg,302245.jpg,INFECTION,1,34d18510f8b10a04cf5481ec469097fe,3083
INFECTION/302246.jpg,302246.jpg,INFECTION,1,1b14d9528decbc4143b484e451778517,3084
INFECTION/302266.jpg,302266.jpg,INFECTION,1,5330f46fb60a2f6e21e3d35d803d4e79,3093
INFECTION/302277.jpg,302277.jpg,INFECTION,1,54c972398cd2e6be3f8f1adf73b4811f,3098
INFECTION/302279.jpg,302279.jpg,INFECTION,1,e5e7449bbfc1fba6f2335d1f6b984a0f,3099
INFECTION/302284.jpg,302284.jpg,INFECTION,1,ac3bba47b8a87c0cd42a6d71aabbac8c,3102
INFECTION/302289.jpg,302289.jpg,INFECTION,1,7f81ef9979a7f3fb7d209b842c625f41,3104
INFECTION/302291.jpg,302291.jpg,INFECTION,1,d3747f780e3dcdf9c5d20b9ba7cf0306,3106
INFECTION/302302.jpg,302302.jpg,INFECTION,1,00e59e794ca3ed23876613ca7d52f2ee,3110
INFECTION/302328.jpg,302328.jpg,INFECTION,1,c549e7db5c42c8cf4da3acf2b7712393,3122
INFECTION/302333.jpg,302333.jpg,INFECTION,1,1837947993b026ad9ccf82eb23a453a4,3124
INFECTION/302338.jpg,302338.jpg,INFECTION,1,8b3f21891c4c515b5258936b688a5db5,3126
INFECTION/302347.jpg,302347.jpg,INFECTION,1,723518fcaf2e352bda8966a0c7a3d063,3130
INFECTION/302352.jpg,302352.jpg,INFECTION,1,e3e8c20689602fe8cd9543d01b73e478,3132
INFECTION/302357.jpg,302357.jpg,INFECTION,1,ac087e0af251d4ca8644df4514f70587,3133
INFECTION/302363.jpg,302363.jpg,INFECTION,1,10f1cb0ad103c25576980e7b859a3c96,3135
INFECTION/302378.jpg,302378.jpg,INFECTION,1,710831c63df013075663a0b60fba5fcf,3139
INFECTION/302393.jpg,302393.jpg,INFECTION,1,25dd1863bd55f3ab91f866d2c89f9b4f,3144
INFECTION/302395.jpg,302395.jpg,INFECTION,1,c99e56db003b38ac033400cae6fef0e5,3145
INFECTION/302416.jpg,302416.jpg,INFECTION,1,2fcee7c59cc1435681a9543053f77f5c,3147
INFECTION/302418.jpg,302418.jpg,INFECTION,1,b33be30bc63ab303c4a69b9d99f11673,3149
INFECTION/302426.jpg,302426.jpg,INFECTION,1,be9e44b3a9f3b38378f75ad625e4b76e,3153
INFECTION/302446.jpg,302446.jpg,INFECTION,1,cfb5117292929626f0319eb6fd981c28,3162
INFECTION/302448.jpg,302448.jpg,INFECTION,1,3e11c29e2a2a1fefe176ac3b88fa4190,3164
INFECTION/302457.jpg,302457.jpg,INFECTION,1,c58ec1839cd999d94eecadefbbfe7aad,3169
INFECTION/302479.jpg,302479.jpg,INFECTION,1,1f9122d966744f516d1d1e86a663839a,3180
INFECTION/302485.jpg,302485.jpg,INFECTION,1,8ea84a0ca745e86357f2a563dc1fcbf2,3183
INFECTION/302486.jpg,302486.jpg,INFECTION,1,fbe1cb6f29803f30c608457900ea12b4,3184
INFECTION/302494.jpg,302494.jpg,INFECTION,1,76616c3c45c570a07aafc63dc71230d9,3188
INFECTION/302497.jpg,302497.jpg,INFECTION,1,aa6e3d6e1c199a14ee500363b7c44157,3189
INFECTION/302500.jpg,302500.jpg,INFECTION,1,478da64c43821b2f7f7edff5c4a06947,3191
INFECTION/302503.jpg,302503.jpg,INFECTION,1,4984eb3069a9a9ef8f492ddebbef0384,3192
INFECTION/302510.jpg,302510.jpg,INFECTION,1,334141ff5e8e78c254d61008e051a298,3197
INFECTION/302540.jpg,302540.jpg,INFECTION,1,17eab7141f858cdc91af57676f05cdec,3210
INFECTION/302546.jpg,302546.jpg,INFECTION,1,e95b7324e68c3c2fedd28e418a334d2c,3211
INFECTION/302547.jpg,302547.jpg,INFECTION,1,b2fcc6fcc774d4d95007ce07b44961ef,3212
INFECTION/302549.jpg,302549.jpg,INFECTION,1,547c81641e9ec9c16131b8cfa7df4583,3213
INFECTION/302554.jpg,302554.jpg,INFECTION,1,2ce21a6daf5e6fc2365b52da66b041a3,3215
INFECTION/302564.jpg,302564.jpg,INFECTION,1,46468b97e5ce221a05b969885c8b838d,3221
INFECTION/302570.jpg,302570.jpg,INFECTION,1,9a0200dbeec5772a87a9d74afbf2ba95,3223
INFECTION/302596.jpg,302596.jpg,INFECTION,1,e90a6b41bfabacb718a79c64b92bf8d0,3237
INFECTION/302599.jpg,302599.jpg,INFECTION,1,219ef4ff4fc5f7d59a348e9e09b26603,3239
INFECTION/302605.jpg,302605.jpg,INFECTION,1,54a04a7baee99afed6f541066612990d,3241
INFECTION/302614.jpg,302614.jpg,INFECTION,1,9255c38423d55f8d591aa3dd763a4cfb,3246
INFECTION/302624.jpg,302624.jpg,INFECTION,1,efa63f7d3f4f2cc234caed7a2ea5f895,3249
INFECTION/302630.jpg,302630.jpg,INFECTION,1,c0bfa9744f2e23ce668e063a9e5f5e97,3252
INFECTION/302637.jpg,302637.jpg,INFECTION,1,fdde41db18cbdf4ade5356e1a2973a02,3255
INFECTION/302686.jpg,302686.jpg,INFECTION,1,7e8272e0202e6267f41535a165a92db3,3275
INFECTION/302695.jpg,302695.jpg,INFECTION,1,cd53dcd3864b9ffee23fe114661ba75a,3278
INFECTION/302697.jpg,302697.jpg,INFECTION,1,59faa58091ca71032c9cb9bb2b6ea758,3279
INFECTION/302699.jpg,302699.jpg,INFECTION,1,bb4889f3394c5adfffa34cdb5e610d65,3281
INFECTION/302725.jpg,302725.jpg,INFECTION,1,d3b6daab51f56a65939299f4031fb6b0,3291
INFECTION/302801.jpg,302801.jpg,INFECTION,1,58995b43f314d29974b05b017cd0a37a,3323
INFECTION/302808.jpg,302808.jpg,INFECTION,1,680155b835f529ca752db26e0e2f5b4c,3325
INFECTION/302825.jpg,302825.jpg,INFECTION,1,408d07fd1f0e510050ad344ebd6659b2,3330
INFECTION/302827.jpg,302827.jpg,INFECTION,1,af6126533f8fa02f905ee5685077f1e7,3331
INFECTION/302829.jpg,302829.jpg,INFECTION,1,bca00be1ca4d5e4a6cf1d0a083037417,3333
INFECTION/302845.jpg,302845.jpg,INFECTION,1,f8cde50fc0e10ecc4160ff3d9e6fcaf3,3338
INFECTION/302912.jpg,302912.jpg,INFECTION,1,0cbad39d0153d234103269c328f59120,3368
INFECTION/302917.jpg,302917.jpg,INFECTION,1,8a473fad3ec7d1edc71848e355d610a8,3371
INFECTION/302919.jpg,302919.jpg,INFECTION,1,bd2ceb1bedf00d8d127bf0ce4bb20299,3372
INFECTION/302920.jpg,302920.jpg,INFECTION,1,f8804998cf3a19eecd939830f1fe942b,3373
INFECTION/302940.jpg,302940.jpg,INFECTION,1,b56c8278f6607fdce1bbbbace2fa6d5f,3382
INFECTION/302947.jpg,302947.jpg,INFECTION,1,032b448a821dc02273ed82395f4fd584,3388
INFECTION/302951.jpg,302951.jpg,INFECTION,1,263d785ad4ca1e9d5e9e1320622911cd,3391
INFECTION/302954.jpg,302954.jpg,INFECTION,1,648ca09fb5e338b62b9637bf02bede9a,3392
INFECTION/302959.jpg,302959.jpg,INFECTION,1,224f5ba099d8d822839a3d037c9a02df,3396
INFECTION/302966.jpg,302966.jpg,INFECTION,1,83f950ae73cdbb00f594e056fdd286f8,3399
INFECTION/302975.jpg,302975.jpg,INFECTION,1,45c2ab3517fff94a5ca552d89daa9f5d,3404
INFECTION/302979.jpg,302979.jpg,INFECTION,1,89ad5cffb55d9a084260c764ca713272,3406
INFECTION/302980.jpg,302980.jpg,INFECTION,1,306f489de3377659c54e3fc9b25ec1fb,3407
INFECTION/302987.jpg,302987.jpg,INFECTION,1,3c6e438f4dae71cbe486016ae89a25ca,3410
INFECTION/303005.jpg,303005.jpg,INFECTION,1,d5784cb5424cd44087e96c9f2e92b1ed,3418
INFECTION/303006.jpg,303006.jpg,INFECTION,1,7db73c713db939ac10286bcac644aa3a,3419
INFECTION/303013.jpg,303013.jpg,INFECTION,1,04bf375e44a37d41e6c74c4702f4af3f,3422
INFECTION/303024.jpg,303024.jpg,INFECTION,1,8fbe8662679c2745879434b45b26475b,3426
INFECTION/303027.jpg,303027.jpg,INFECTION,1,53c88c611d4ad45c2eab63062da359a4,3427
INFECTION/303040.jpg,303040.jpg,INFECTION,1,08301b62440fa30362e1b3649a7e4e4e,3432
INFECTION/303044.jpg,303044.jpg,INFECTION,1,b90bdc961ed9e863bf1d6e0ffcb05446,3435
INFECTION/303062.jpg,303062.jpg,INFECTION,1,45f737e6dc402e5838cc7fea4c4b3c66,3443
INFECTION/303066.jpg,303066.jpg,INFECTION,1,77eac3720569d3b01ed8443298f2c3d7,3446
INFECTION/303067.jpg,303067.jpg,INFECTION,1,eb482f16d5f2f8e096aadb1d81773ecd,3447
INFECTION/303082.jpg,303082.jpg,INFECTION,1,7497e0769d9a1084f60e4d5cd1443a21,3454
INFECTION/303090.jpg,303090.jpg,INFECTION,1,2ffffa3a3f3fc9a9f5fcb76817c243b3,3456
INFECTION/303091.jpg,303091.jpg,INFECTION,1,9d641047f04c7ba9383963ff633a39df,3457
INFECTION/303092.jpg,303092.jpg,INFECTION,1,4e21fa1dfdbf906c912ae7920b937e97,3458
INFECTION/303120.jpg,303120.jpg,INFECTION,1,03c254f701320fe196fc280a44d3498b,3473
INFECTION/303127.jpg,303127.jpg,INFECTION,1,f1ed3b5e8650b5032916241e5b44b2c1,3475
INFECTION/303138.jpg,303138.jpg,INFECTION,1,c1fe1935093e56438e56a8dbe2c9a42d,3481
INFECTION/303141.jpg,303141.jpg,INFECTION,1,d050e62f5535f126d339c166d275d2e3,3482
INFECTION/303146.jpg,303146.jpg,INFECTION,1,afb4cfc9cddb1d6a33fb3e122010bd6b,3485
INFECTION/303148.jpg,303148.jpg,INFECTION,1,7029443e03fde3b80381018f4f369c85,3487
INFECTION/303158.jpg,303158.jpg,INFECTION,1,c9f49f4b0927ddca7af0adec4544126d,3492
INFECTION/303166.jpg,303166.jpg,INFECTION,1,b5a526edc7f9ca4ce44ead6ecd07ebe5,3494
INFECTION/303168.jpg,303168.jpg,INFECTION,1,871c3e944068405510eaa5b8d30c7094,3495
INFECTION/303179.jpg,303179.jpg,INFECTION,1,a3c1dae9a86f9600486570e149407dea,3500
INFECTION/303204.jpg,303204.jpg,INFECTION,1,d7b0d3aec502a80ddddf613fc0403ec9,3511
INFECTION/303224.jpg,303224.jpg,INFECTION,1,0cefc2395faa3c95dd08971e6209d60e,3517
INFECTION/303228.jpg,303228.jpg,INFECTION,1,99f1823eb926807f75f0abc1fa865bff,3519
INFECTION/303235.jpg,303235.jpg,INFECTION,1,aa935626f13e449cc4ac6947995dbd3c,3522
INFECTION/303240.jpg,303240.jpg,INFECTION,1,c96a0faf6cf82ff317ad0a472dcaaa5b,3524
INFECTION/303250.jpg,303250.jpg,INFECTION,1,fe57bd2b05b3a4f9d4ce2d764629ac40,3528
INFECTION/303255.jpg,303255.jpg,INFECTION,1,e04da36152cd6a1095c60544d46d028e,3531
INFECTION/303257.jpg,303257.jpg,INFECTION,1,b1cc9b67cb5a58084bc129ccd3ad63bf,3532
INFECTION/303261.jpg,303261.jpg,INFECTION,1,d27d766fe7dfe086380f8a878335f290,3533
INFECTION/303275.jpg,303275.jpg,INFECTION,1,31dfdfeef09ab616a7b5e996683f2675,3536
INFECTION/303278.jpg,303278.jpg,INFECTION,1,eaa21ca85c11e5ed676a59561526adfe,3537
INFECTION/303286.jpg,303286.jpg,INFECTION,1,f9596683a524b78c492657b2302f1bfc,3539
INFECTION/303291.jpg,303291.jpg,INFECTION,1,d1cad937a01a087b7231cc03d3e12afb,3541
INFECTION/303305.jpg,303305.jpg,INFECTION,1,1b2ab35dd4a959b636bd5935c84106e3,3549
INFECTION/303333.jpg,303333.jpg,INFECTION,1,b29b0193442b98b749700198bdd03a62,3561
INFECTION/303344.jpg,303344.jpg,INFECTION,1,8c69e41c03888059e4b5b9a3792186f6,3563
INFECTION/303352.jpg,303352.jpg,INFECTION,1,2eac0c8967b7fa56ec02a83b6e803cab,3564
INFECTION/303354.jpg,303354.jpg,INFECTION,1,b67d52f5003430e3ca9d05e965b63be3,3565
INFECTION/303355.jpg,303355.jpg,INFECTION,1,b02410987cbca924d845947415abe07a,3566
INFECTION/303371.jpg,303371.jpg,INFECTION,1,58253d1dd124600e9221817cf176c03c,3571
INFECTION/303397.jpg,303397.jpg,INFECTION,1,02d2ffbcc97431294bc33e7514b26e57,3583
INFECTION/303427.jpg,303427.jpg,INFECTION,1,a2837f7a15a2e46424dda1e51d5eb3e1,3595
INFECTION/303466.jpg,303466.jpg,INFECTION,1,f80ae843a8bc1fa360bbd32767924a92,3610
INFECTION/303483.jpg,303483.jpg,INFECTION,1,6bcabd6e6ab69feaea8404b6f3e3d6a4,3619
INFECTION/303486.jpg,303486.jpg,INFECTION,1,8e0e9b818163c21977b10caa5f8399ea,3622
INFECTION/303510.jpg,303510.jpg,INFECTION,1,ff45f972111db530ca592d9d7916fb08,3630
INFECTION/303518.jpg,303518.jpg,INFECTION,1,1cdc534568959a8f25fdf21fbeceaa06,3632
INFECTION/303525.jpg,303525.jpg,INFECTION,1,64bfee8bccf1e32b3873cb0f253e53c2,3637
INFECTION/303534.jpg,303534.jpg,INFECTION,1,4058ec61a610086ccd6e92f36346a52d,3641
INFECTION/303538.jpg,303538.jpg,INFECTION,1,2b8e0e4b533c6bc03bb14dcfc760586d,3643
INFECTION/303540.jpg,303540.jpg,INFECTION,1,7aaebe6c7086b245f1a35f887e56de13,3645
INFECTION/303550.jpg,303550.jpg,INFECTION,1,5e7b19d79d3e01d99c7e1ef9622bf49b,3650
INFECTION/303553.jpg,303553.jpg,INFECTION,1,b1eff484d1f639effcf783a092b6ff39,3651
INFECTION/303565.jpg,303565.jpg,INFECTION,1,a52b4f815d830f337d9a676bb8ea8709,3657
INFECTION/303572.jpg,303572.jpg,INFECTION,1,5432ad0fc6c8d7e9008b32e89f16b24c,3659
INFECTION/303581.jpg,303581.jpg,INFECTION,1,2ab40869ed3c01d1da7d6d573a05b858,3662
INFECTION/303610.jpg,303610.jpg,INFECTION,1,8f7185305c4c975f3470360a3ed6b2c0,3676
INFECTION/303611.jpg,303611.jpg,INFECTION,1,dfb413c633a0b4cdd100e261f1b9ab51,3677
INFECTION/303616.jpg,303616.jpg,INFECTION,1,e707da62acc0bfaf7c173fe63c96e047,3679
INFECTION/303627.jpg,303627.jpg,INFECTION,1,93dbfc49d3980517544ed110e5ad691d,3684
INFECTION/303637.jpg,303637.jpg,INFECTION,1,cdb4feff31a275e2b97392a88c3f8b80,3688
INFECTION/303643.jpg,303643.jpg,INFECTION,1,9c6511eb33da83af71e81b6f5ded6021,3690
INFECTION/303652.jpg,303652.jpg,INFECTION,1,129f8556eb05413649ad1469976aba0f,3692
INFECTION/303653.jpg,303653.jpg,INFECTION,1,3662b54793ea06690d7ffe72de641655,3693
INFECTION/303655.jpg,303655.jpg,INFECTION,1,1b9092bcf59d1394596b2aa255c74820,3695
INFECTION/303673.jpg,303673.jpg,INFECTION,1,45bb17b0a5e9a53746e5a93b202288a2,3704
INFECTION/303676.jpg,303676.jpg,INFECTION,1,b0213c93db3e70576d2bc6ee64260b6f,3706
INFECTION/303688.jpg,303688.jpg,INFECTION,1,bd4fb2a8dbd5ad720d205e0da7010cec,3707
INFECTION/303698.jpg,303698.jpg,INFECTION,1,813b9b0c122a9637f1590c7303697c09,3711
INFECTION/303720.jpg,303720.jpg,INFECTION,1,791ce7af3e9c09e1550b7905e0df2cf6,3721
INFECTION/303724.jpg,303724.jpg,INFECTION,1,5f7c9c0923146df6a0c12b885ab52df4,3723
INFECTION/303734.jpg,303734.jpg,INFECTION,1,ba4a97703cc0ae9b42e8c42fcc713a87,3729
INFECTION/303755.jpg,303755.jpg,INFECTION,1,d50c3f2fda70b6c731df2cc7c4f52903,3737
INFECTION/303786.jpg,303786.jpg,INFECTION,1,95cb2abf6ac6450e1603cd5a7005a8ff,3753
INFECTION/303802.jpg,303802.jpg,INFECTION,1,938b2a52fe090059375faedb63f208a8,3757
INFECTION/303803.jpg,303803.jpg,INFECTION,1,4f5f6b8accaf0e73e0a0c55d73cad7ce,3758
INFECTION/303818.jpg,303818.jpg,INFECTION,1,a25b2ef71936019d9a2917a85f67f849,3765
INFECTION/303843.jpg,303843.jpg,INFECTION,1,8a584abdc653363db136c25c4a3f7489,3778
INFECTION/303858.jpg,303858.jpg,INFECTION,1,35081eca904cd2849537b80b770175a3,3785
INFECTION/303879.jpg,303879.jpg,INFECTION,1,31e0ecd4d182f291b4de330923192573,3793
INFECTION/303882.jpg,303882.jpg,INFECTION,1,52521e4c9497e98b80a88ed3d3bf3031,3795
INFECTION/303899.jpg,303899.jpg,INFECTION,1,1c1888e93ef3f1fecd333ff5ff9c9a60,3801
INFECTION/303901.jpg,303901.jpg,INFECTION,1,4c47fc2aac863ed6602097c7012d585e,3803
INFECTION/303957.jpg,303957.jpg,INFECTION,1,7a975e32517e5e3f0f8416e5cae47dd8,3829
INFECTION/303958.jpg,303958.jpg,INFECTION,1,7bf2e07c5509a8964c62f10c6ce62b47,3830
INFECTION/303966.jpg,303966.jpg,INFECTION,1,696f842c9c69683802d28bb52a1fe4f9,3835
INFECTION/303995.jpg,303995.jpg,INFECTION,1,9f1b0459033ea9c7d037a54ec245778a,3850
INFECTION/303997.jpg,303997.jpg,INFECTION,1,c271c1638ff394778d5900a5e4060105,3851
INFECTION/304047.jpg,304047.jpg,INFECTION,1,182400cd202ec4b5a4136eecf632f396,3866
INFECTION/304051.jpg,304051.jpg,INFECTION,1,5d25bef678a79647ca15e20f24a56b9b,3869
INFECTION/304058.jpg,304058.jpg,INFECTION,1,322d3508f977ce7dd48f1512ea9dad61,3872
INFECTION/304069.jpg,304069.jpg,INFECTION,1,649e4a6aaffe059268cfe2ace87262e7,3877
INFECTION/304071.jpg,304071.jpg,INFECTION,1,ee3ae73e7933956e6a63af37696ff655,3878
INFECTION/304073.jpg,304073.jpg,INFECTION,1,f9aeaf1e23defe3b3635c3edb3e68a64,3879
INFECTION/304088.jpg,304088.jpg,INFECTION,1,046dde3b722e35a574a52fb8869b28f3,3888
INFECTION/304089.jpg,304089.jpg,INFECTION,1,bc8d9e8120913687dae4ef8fde604f29,3889
INFECTION/304105.jpg,304105.jpg,INFECTION,1,918399e0ff5b4c3621b375193be4b90c,3899
INFECTION/304110.jpg,304110.jpg,INFECTION,1,2bf5cd4753d3792c717cdddec2122af7,3900
INFECTION/304123.jpg,304123.jpg,INFECTION,1,ef88cdff0d486f612495c1b13f2575d9,3908
INFECTION/304136.jpg,304136.jpg,INFECTION,1,db7bcb542ccf76579e5bd020165a109b,3917
INFECTION/304138.jpg,304138.jpg,INFECTION,1,d43ebdb03d77d10fc8c642a599a4a1cd,3918
INFECTION/304141.jpg,304141.jpg,INFECTION,1,da899968f88b0166607a2dd23d6a8c0d,3919
INFECTION/304160.jpg,304160.jpg,INFECTION,1,25f4b09ea0392254d166bbd04a4b4b41,3924
INFECTION/304163.jpg,304163.jpg,INFECTION,1,7d5b50ada0246db43ffa8ce46fa9940e,3925
INFECTION/304170.jpg,304170.jpg,INFECTION,1,706209782989262e310f3d6801e66444,3929
INFECTION/304186.jpg,304186.jpg,INFECTION,1,9ed951dfe8c3566ead383934bb251256,3936
INFECTION/304188.jpg,304188.jpg,INFECTION,1,004bca934d757fe57f580f0152b127f4,3938
INFECTION/304189.jpg,304189.jpg,INFECTION,1,a35b359ef83495a1a3810b4cc10b8292,3939
INFECTION/304205.jpg,304205.jpg,INFECTION,1,6b675af9316d788253340c59c8b28b9a,3947
INFECTION/304214.jpg,304214.jpg,INFECTION,1,46bbb3e6e5c52ec688fc624c3c583595,3950
INFECTION/304223.jpg,304223.jpg,INFECTION,1,d210ed192df7a15c9ddf362b516fa842,3956
INFECTION/304260.jpg,304260.jpg,INFECTION,1,c8d3ad9a54dd1e3a6211e412bb1f7b31,3970
INFECTION/304274.jpg,304274.jpg,INFECTION,1,3953a5d1f7c89a0e4a185697081bbd8b,3974
INFECTION/304291.jpg,304291.jpg,INFECTION,1,953799e989e262822890e2ca2790798f,3981
INFECTION/304298.jpg,304298.jpg,INFECTION,1,a72b8c5046480f87242bbfb97eaa2697,3983
INFECTION/304319.jpg,304319.jpg,INFECTION,1,2f442d94bdb6085c295debd037bec6f7,3992
INFECTION/304337.jpg,304337.jpg,INFECTION,1,a25921f7a6ad90957aecda0fce7d0a59,3999
INFECTION/304342.jpg,304342.jpg,INFECTION,1,5cbb0c9e30d39c600cea38658089d7fa,4001
INFECTION/304343.jpg,304343.jpg,INFECTION,1,9e02beaf0d5a4b36b9d1afab7fb38821,4002
INFECTION/304353.jpg,304353.jpg,INFECTION,1,5b25fb55fb8eaf532e7bd5a7e7f8ac2e,4007
INFECTION/304358.jpg,304358.jpg,INFECTION,1,a3807a0fb2906d3d9ab41e9717d5cf7b,4009
INFECTION/304365.jpg,304365.jpg,INFECTION,1,77019afe0d126b9524208bf56be80bdc,4012
INFECTION/304377.jpg,304377.jpg,INFECTION,1,0f96f33b8b94f8551206797d1d2616bf,4016
INFECTION/304381.jpg,304381.jpg,INFECTION,1,34e6d4d92cd7b3039f1d6c7782eb47f5,4018
INFECTION/304401.jpg,304401.jpg,INFECTION,1,9c56bc0e430179c69828d1d60f44edcd,4025
INFECTION/304414.jpg,304414.jpg,INFECTION,1,0b066fdd4abdde48d4803d38544b0378,4030
INFECTION/304441.jpg,304441.jpg,INFECTION,1,80ed14eb31faa96781d449a611be3f62,4042
INFECTION/304447.jpg,304447.jpg,INFECTION,1,d4fe557a54da1829f282fb0b82a30fa7,4044
INFECTION/304462.jpg,304462.jpg,INFECTION,1,e6ea6f37385e6bc1b60dd99db132ff5e,4047
INFECTION/304497.jpg,304497.jpg,INFECTION,1,d769468d5b92ef44af706db28859fdcd,4067
INFECTION/304501.jpg,304501.jpg,INFECTION,1,12dca2e1e11ba11bb153965340a32dd6,4069
INFECTION/304522.jpg,304522.jpg,INFECTION,1,94fec4c3e299130ba527b113b81fa6ed,4080
INFECTION/304524.jpg,304524.jpg,INFECTION,1,95041383f33b217ac2b199e5ecd9f628,4081
INFECTION/304525.jpg,304525.jpg,INFECTION,1,5cb096d0c4df69bd5c09694468718454,4082
INFECTION/304526.jpg,304526.jpg,INFECTION,1,701312a8c590b39c02a0443fee05e69e,4083
INFECTION/304547.jpg,304547.jpg,INFECTION,1,c3de89e11ce1bdd3497240ca63155a2b,4089
INFECTION/304573.jpg,304573.jpg,INFECTION,1,898ed0245a97a75143251b2eb1cd95da,4099
INFECTION/304607.jpg,304607.jpg,INFECTION,1,da36feeb0772d2a7de03192ddfa46202,4112
INFECTION/304622.jpg,304622.jpg,INFECTION,1,a579ceb21f383c05ea876e2a32d4310c,4117
INFECTION/304636.jpg,304636.jpg,INFECTION,1,570242938bf5c0306523da3882a300c0,4124
INFECTION/304637.jpg,304637.jpg,INFECTION,1,5fb1a2f55f6c9c9d644cb8003659e016,4125
INFECTION/304639.jpg,304639.jpg,INFECTION,1,0812ed8f3b551b96c8a997bf50687e9d,4126
INFECTION/304648.jpg,304648.jpg,INFECTION,1,ba243227da69b6cffd44e9815356daf7,4129
INFECTION/304654.jpg,304654.jpg,INFECTION,1,82058e5a30809861c2858fb3c63ba15a,4131
INFECTION/304663.jpg,304663.jpg,INFECTION,1,4011b37784bbf272c18a4648e2bd7456,4134
INFECTION/304669.jpg,304669.jpg,INFECTION,1,ead34c6b4b74e95926194e5c7cd3c7d4,4137
INFECTION/304691.jpg,304691.jpg,INFECTION,1,ef332e66cdecde07ed8be08b57dd7828,4146
INFECTION/304697.jpg,304697.jpg,INFECTION,1,d6d397fa16a2fd38869a0f4881615346,4148
INFECTION/304709.jpg,304709.jpg,INFECTION,1,f1b7b99f9fd413b29f03043292be3f9f,4154
INFECTION/304749.jpg,304749.jpg,INFECTION,1,71d23ea9dbd0fed438b43d5222b0fcb6,4171
INFECTION/304754.jpg,304754.jpg,INFECTION,1,c36538d963fc7e16465069ce3a383867,4173
INFECTION/304761.jpg,304761.jpg,INFECTION,1,2f470453367fded6e57898ed16e25dc2,4177
INFECTION/304775.jpg,304775.jpg,INFECTION,1,9d40309e1ae072492f9c159988adafd3,4185
INFECTION/304790.jpg,304790.jpg,INFECTION,1,c03444b7d6a1749225a2b6535398657e,4193
INFECTION/304791.jpg,304791.jpg,INFECTION,1,15174eacb099233540da820b8db372b2,4194
INFECTION/304794.jpg,304794.jpg,INFECTION,1,31fbda4380c251c477d8de5839138726,4196
INFECTION/304800.jpg,304800.jpg,INFECTION,1,0a30f84eacd4414b5a7932a818a16019,4201
INFECTION/304805.jpg,304805.jpg,INFECTION,1,df1baf588cb2b623e08cc284c8b379a6,4203
INFECTION/304810.jpg,304810.jpg,INFECTION,1,206b393bc107f62ac749d632b82640c0,4207
INFECTION/304821.jpg,304821.jpg,INFECTION,1,41c84900be88cf9b1e67014dcc7537eb,4211
INFECTION/304867.jpg,304867.jpg,INFECTION,1,c738d3a8df140b69199a345a281cec92,4226
INFECTION/304871.jpg,304871.jpg,INFECTION,1,094e21983ba9af4bff0d797df2f59d2c,4229
INFECTION/304873.jpg,304873.jpg,INFECTION,1,60587ba8b4acd41f2746da1f7ffd3a34,4230
INFECTION/304876.jpg,304876.jpg,INFECTION,1,d23189e3c1602806d5dddfb2aadbeb6f,4233
INFECTION/304878.jpg,304878.jpg,INFECTION,1,279be90b37ed8a1fa7d4bdcc557b206f,4234
INFECTION/304880.jpg,304880.jpg,INFECTION,1,e635b3a2cb74ddc5c3e7f1fe479c252c,4235
INFECTION/304889.jpg,304889.jpg,INFECTION,1,94ab2cf2b204c16f52eab5cc15032a73,4239
INFECTION/304898.jpg,304898.jpg,INFECTION,1,8db1875be98f433ca9977125d6b888d6,4245
INFECTION/304912.jpg,304912.jpg,INFECTION,1,a5e13d9533188d135b5868753f7bc3b9,4250
INFECTION/304925.jpg,304925.jpg,INFECTION,1,551f242b38f1b81e5c6361e52403fefb,4256
INFECTION/304939.jpg,304939.jpg,INFECTION,1,54792c92fbff9ec0b1bc5fb68a7f5780,4262
INFECTION/304943.jpg,304943.jpg,INFECTION,1,34b1db61041dc1c1219c7e7ed4916295,4264
INFECTION/304947.jpg,304947.jpg,INFECTION,1,f8a1dfe04fac19045d73897e2ac041cc,4267
INFECTION/304969.jpg,304969.jpg,INFECTION,1,7d5ab102c495ac04a09190d69e5b6a15,4271
INFECTION/304994.jpg,304994.jpg,INFECTION,1,f76fd2df22ecd5105113652c14b1fb90,4279
INFECTION/305005.jpg,305005.jpg,INFECTION,1,ed581a6159d5dd7167fa61a15d7be3f1,4283
INFECTION/305011.jpg,305011.jpg,INFECTION,1,04e66885f4f749f5b761c4f743742e6f,4285
INFECTION/305053.jpg,305053.jpg,INFECTION,1,0575d5e6e9f64f695da7cdbaa3035ee8,4301
INFECTION/305060.jpg,305060.jpg,INFECTION,1,21651e4d820152fb8a6651cc40c6a398,4302
INFECTION/305065.jpg,305065.jpg,INFECTION,1,333754d596afd9b422d30f0fd57aa449,4305
INFECTION/305074.jpg,305074.jpg,INFECTION,1,fb5de4b052d8ed4ad3c62cb93dfa9445,4310
INFECTION/305080.jpg,305080.jpg,INFECTION,1,8e6016a436f3fbff87ab0393ee16d20e,4312
INFECTION/305086.jpg,305086.jpg,INFECTION,1,b9f6ef58ac55ef0ba6d834c6fad8a34b,4314
INFECTION/305121.jpg,305121.jpg,INFECTION,1,1eacf49b3147d774cb5803c349b3424a,4326
INFECTION/305122.jpg,305122.jpg,INFECTION,1,7cff0ee93ffc8ad24c9082968e9bd7e5,4327
INFECTION/305130.jpg,305130.jpg,INFECTION,1,037650063019c9ed84e79967c807c5a1,4330
INFECTION/305135.jpg,305135.jpg,INFECTION,1,6a316b02a9cc5c06ca37d9d87d3cd5c4,4333
INFECTION/305144.jpg,305144.jpg,INFECTION,1,0867120cf8d73182c7ad671f4d625e13,4335
INFECTION/305149.jpg,305149.jpg,INFECTION,1,b4809d9e0a1c73eff1618509ecdf22bb,4339
INFECTION/305154.jpg,305154.jpg,INFECTION,1,210dc1a97c9ae2850bc24453a5e694ce,4342
INFECTION/305163.jpg,305163.jpg,INFECTION,1,2160e2422775b6b369b50425d61b223a,4344
INFECTION/305169.jpg,305169.jpg,INFECTION,1,0a21a74710f0dddfb1aa745b96b71427,4346
INFECTION/305174.jpg,305174.jpg,INFECTION,1,1e97034fe3d0047fb3eef403a38dbc15,4347
INFECTION/305180.jpg,305180.jpg,INFECTION,1,76b278a735515ca3f98d7995ffe3aa5d,4351
INFECTION/305185.jpg,305185.jpg,INFECTION,1,48fc8628b927613966d5d5acb2dab574,4352
INFECTION/305213.jpg,305213.jpg,INFECTION,1,98b9c6e3ea63548b841168c079fb72f6,4362
INFECTION/305216.jpg,305216.jpg,INFECTION,1,f7dd864b76ca87d377a9f3e6e66c7670,4363
INFECTION/305221.jpg,305221.jpg,INFECTION,1,3b2c115f335401adf4ec809c09fac2a5,4364
INFECTION/305224.jpg,305224.jpg,INFECTION,1,7d8316f3d134ce2632f3c1941725e8b8,4366
INFECTION/305241.jpg,305241.jpg,INFECTION,1,248cf069aa42789e02311a423089737a,4373
INFECTION/305252.jpg,305252.jpg,INFECTION,1,1aad450f109502e6dd1140568fec2a1e,4380
INFECTION/305257.jpg,305257.jpg,INFECTION,1,0a3a7147613fe8a15aeab079ced26707,4381
INFECTION/305259.jpg,305259.jpg,INFECTION,1,73e745f743cd7dc1349c6eeddfe848d1,4382
INFECTION/305271.jpg,305271.jpg,INFECTION,1,0db7f8216e2a9e21ca2108ee50620523,4388
INFECTION/305275.jpg,305275.jpg,INFECTION,1,a2c91ddf09e4a8baeb875743a8a4ab21,4390
INFECTION/305308.jpg,305308.jpg,INFECTION,1,00b06b90c2fe7092f2ef074823fe5015,4398
INFECTION/305309.jpg,305309.jpg,INFECTION,1,eb8deb3b434f4acd632c8876a5d8ab52,4399
INFECTION/305320.jpg,305320.jpg,INFECTION,1,5acfb438a544e9a791352f434fc053b4,4403
INFECTION/305321.jpg,305321.jpg,INFECTION,1,5ef3a4dff9817e6c314a794fa6005b53,4404
INFECTION/305323.jpg,305323.jpg,INFECTION,1,513d77eea0558666dc4e5e0145b2fe96,4406
INFECTION/305340.jpg,305340.jpg,INFECTION,1,91d131b9cc6e5c5bd1c09bb3a52366ab,4413
INFECTION/305383.jpg,305383.jpg,INFECTION,1,e6813d7300abdce3da27b172df14fe51,4427
INFECTION/305397.jpg,305397.jpg,INFECTION,1,6751d7995e04707a5bb548d8c650d39d,4432
INFECTION/305406.jpg,305406.jpg,INFECTION,1,d37c5844f557e9ed0fc37565f40aae9f,4437
INFECTION/305460.jpg,305460.jpg,INFECTION,1,214b8dc3ea2fe21f3c4ff94bda97b290,4448
INFECTION/305464.jpg,305464.jpg,INFECTION,1,f53ad595bf1a15379ed37dfe8bd816e3,4450
INFECTION/305468.jpg,305468.jpg,INFECTION,1,4efad07d7248f130723a218bd822d819,4451
INFECTION/305470.jpg,305470.jpg,INFECTION,1,04b1b58710dc61437fbc9222043294ad,4452
INFECTION/305471.jpg,305471.jpg,INFECTION,1,6a15cfdfc81d51280c02288dfb175cd1,4453
INFECTION/305474.jpg,305474.jpg,INFECTION,1,6ae764710afd25f6e617fef1d2e55cdf,4455
INFECTION/305479.jpg,305479.jpg,INFECTION,1,ee17a141ebdd25976b3762bf17a99d06,4456
INFECTION/305495.jpg,305495.jpg,INFECTION,1,c6144bfbce288e0dc2366897ac1c410b,4462
INFECTION/305503.jpg,305503.jpg,INFECTION,1,4908823a0bb9c5b8469e308d18f7979a,4467
INFECTION/305537.jpg,305537.jpg,INFECTION,1,21dd38f08c60431fed5d28035377e381,4480
INFECTION/305538.jpg,305538.jpg,INFECTION,1,86695a542b690c0a1469d9542e3030d8,4481
INFECTION/305539.jpg,305539.jpg,INFECTION,1,47f6a281cf8e87aa15fa83897fb0fd71,4482
INFECTION/305541.jpg,305541.jpg,INFECTION,1,1e436a2abfa4deee856264ac3d97c462,4483
INFECTION/305548.jpg,305548.jpg,INFECTION,1,30051045bd4303c08247a46d785df963,4488
INFECTION/305554.jpg,305554.jpg,INFECTION,1,f2948a4df4f8fb2447361ab2ef372f74,4490
INFECTION/305558.jpg,305558.jpg,INFECTION,1,856d3a0bdcb2efcdd30326d71c358bac,4491
INFECTION/305569.jpg,305569.jpg,INFECTION,1,97e61ebc870fc17b1f9dd318880b9852,4496
INFECTION/305589.jpg,305589.jpg,INFECTION,1,70280b42442381404d6094ce8c3b8a50,4504
INFECTION/305592.jpg,305592.jpg,INFECTION,1,c5b8aea973fcb4f1669f1511cdb074f2,4505
INFECTION/305595.jpg,305595.jpg,INFECTION,1,2f930f94ddcf7c4d1b9b8dde81600ad6,4508
INFECTION/305598.jpg,305598.jpg,INFECTION,1,dd28e530695f27f9275502b751a3d668,4510
INFECTION/305612.jpg,305612.jpg,INFECTION,1,6e65c240f99e9e0b45fd4b13b98133f8,4519
INFECTION/305614.jpg,305614.jpg,INFECTION,1,e0eb09907d24efdc311327110ffd05e0,4520
INFECTION/305625.jpg,305625.jpg,INFECTION,1,1ae5ae571fec7600d8bed5981ca57961,4526
INFECTION/305635.jpg,305635.jpg,INFECTION,1,1689000505f7f1ce47fbebfc35040ebb,4531
INFECTION/305648.jpg,305648.jpg,INFECTION,1,9ece4f981e6b24dd5ab63e859e2e59a9,4538
INFECTION/305660.jpg,305660.jpg,INFECTION,1,ef8752da1ea346a97698ceec53e0f02d,4543
INFECTION/305686.jpg,305686.jpg,INFECTION,1,a245f402ffb817499ae86cad4093f811,4553
INFECTION/305694.jpg,305694.jpg,INFECTION,1,e6bde713756c41b3726f803bc4386d20,4555
INFECTION/305707.jpg,305707.jpg,INFECTION,1,fc422fe7012da0a19a81baf4f06d3f31,4559
INFECTION/305713.jpg,305713.jpg,INFECTION,1,b2044c8930a8d8b391957ee4608be2b3,4562
INFECTION/305728.jpg,305728.jpg,INFECTION,1,e03237665485e4d8c95e3338b51c2a8b,4566
INFECTION/305744.jpg,305744.jpg,INFECTION,1,1e6f0dc2686400ee62bbf89975e038a2,4571
INFECTION/305769.jpg,305769.jpg,INFECTION,1,de6cd0fd30b88f8166f166b2c4c07bea,4575
INFECTION/305778.jpg,305778.jpg,INFECTION,1,67e677fea63a90329060f0ab75f15783,4580
INFECTION/305796.jpg,305796.jpg,INFECTION,1,1eb1f17d1436b213c6e672f65359d6cf,4588
INFECTION/305809.jpg,305809.jpg,INFECTION,1,666e45528d8abf1d2cf529b295bb9557,4596
INFECTION/305813.jpg,305813.jpg,INFECTION,1,290e0fc0759c5bf74e023698735bb335,4598
INFECTION/305817.jpg,305817.jpg,INFECTION,1,d8a86fbce976c0ee50f2ebffececd0ca,4600
INFECTION/305852.jpg,305852.jpg,INFECTION,1,c276769c5e0af7c8a8dc181a9bf1630f,4618
INFECTION/305856.jpg,305856.jpg,INFECTION,1,60c8e1a3255d6aec888de22f587f76a3,4620
INFECTION/305860.jpg,305860.jpg,INFECTION,1,73d56be706b23b547c6d3ed8a505f54c,4621
INFECTION/305866.jpg,305866.jpg,INFECTION,1,5686dfc586747d0804d3140a8b7cb9ce,4625
INFECTION/305871.jpg,305871.jpg,INFECTION,1,4a6bc682e2e912fc8794ce0916839bb8,4627
INFECTION/305885.jpg,305885.jpg,INFECTION,1,8d5e2d9f707b13624438cb0a376db252,4631
INFECTION/305894.jpg,305894.jpg,INFECTION,1,0e2b86c96c73f6bc9741a0e627d10b48,4637
INFECTION/305902.jpg,305902.jpg,INFECTION,1,538f29d5c239210ae45b4541b735b934,4642
INFECTION/305920.jpg,305920.jpg,INFECTION,1,9ca6d2d1063f1201e2a2ed0220f4d256,4648
INFECTION/305921.jpg,305921.jpg,INFECTION,1,b8b2b29f67ca835fd829306f90ef57f5,4649
INFECTION/305935.jpg,305935.jpg,INFECTION,1,36528e48f05626cd5fd3334427536667,4655
INFECTION/305939.jpg,305939.jpg,INFECTION,1,24f99b5c68f8c1808640f87fbd93d3f8,4657
INFECTION/305958.jpg,305958.jpg,INFECTION,1,7533e3f49f0530246440424fc3e4ba23,4665
INFECTION/305982.jpg,305982.jpg,INFECTION,1,c5a52c2f4efa274d25700198f59e956e,4675
INFECTION/305987.jpg,305987.jpg,INFECTION,1,082512f2a355fa7a0b3f022daa42b3a1,4677
INFECTION/305988.jpg,305988.jpg,INFECTION,1,ad9311f9eec3f5d1844034e465914dca,4678
INFECTION/306001.jpg,306001.jpg,INFECTION,1,6f3432bb5624fc030fac4a5196b91b77,4688
INFECTION/306011.jpg,306011.jpg,INFECTION,1,86217ac18ddd56ed31eab6eaefde6d59,4694
INFECTION/306026.jpg,306026.jpg,INFECTION,1,65d9173f5e70347989377b7a52744207,4701
INFECTION/306048.jpg,306048.jpg,INFECTION,1,3ac1e678a017a85e3fe89dd173c9ae90,4712
INFECTION/306055.jpg,306055.jpg,INFECTION,1,64517dc661068eaf4551bec9b4df43a2,4714
INFECTION/306075.jpg,306075.jpg,INFECTION,1,0c3d2c488d40e9a2d7514a2a8b323f9a,4722
INFECTION/306077.jpg,306077.jpg,INFECTION,1,b195566438b1d000c1ca959259fd5987,4723
INFECTION/306101.jpg,306101.jpg,INFECTION,1,833b83b15ffe1712f28f6fd66c87fb2e,4730
INFECTION/306102.jpg,306102.jpg,INFECTION,1,152859960894a3ffda776486b8b1e490,4731
INFECTION/306109.jpg,306109.jpg,INFECTION,1,e65809863be870091f4362ac834feb2d,4736
INFECTION/306117.jpg,306117.jpg,INFECTION,1,dabad9ff846e2d39e1a181173690f7bd,4741
INFECTION/306118.jpg,306118.jpg,INFECTION,1,b3b696f2af9066c527a43999adbd884c,4742
INFECTION/306164.jpg,306164.jpg,INFECTION,1,9565bcebabdc7c6863acdd2cc08e5b22,4755
INFECTION/306172.jpg,306172.jpg,INFECTION,1,39df8a9939ee2f233f0a517bf6950344,4758
INFECTION/306177.jpg,306177.jpg,INFECTION,1,9ca4133c0ee7625df02662b273433b71,4760
INFECTION/306180.jpg,306180.jpg,INFECTION,1,b6762a905e79ddcad55d3f7cc73ee81a,4762
INFECTION/306181.jpg,306181.jpg,INFECTION,1,eca8200e7f5377629b065101b19690c7,4763
INFECTION/306246.jpg,306246.jpg,INFECTION,1,d5d634f259c0406cb2b7c61066b022dd,4791
INFECTION/306256.jpg,306256.jpg,INFECTION,1,2d37bebbbcdb98025093dcc31b8d9d5d,4795
INFECTION/306262.jpg,306262.jpg,INFECTION,1,7b82e879c1a28fc192b59b3f64fac772,4798
INFECTION/306266.jpg,306266.jpg,INFECTION,1,3cb2b47dc8760e9d0cbd61bc620cc158,4801
INFECTION/306272.jpg,306272.jpg,INFECTION,1,60f6976f1e5a48856c674e4cd3edc577,4804
INFECTION/306286.jpg,306286.jpg,INFECTION,1,05dc599825ae13b04221d4e2211ebffa,4810
INFECTION/306292.jpg,306292.jpg,INFECTION,1,5d0dd59dd56361c0958d37264712840b,4812
INFECTION/306295.jpg,306295.jpg,INFECTION,1,8040e6fbaa45112cf35701a1d9a5afdc,4813
INFECTION/306307.jpg,306307.jpg,INFECTION,1,610eae487e0d128124aa683183e58892,4819
INFECTION/306317.jpg,306317.jpg,INFECTION,1,78c46f37d043540709637abc38ff94ec,4824
INFECTION/306320.jpg,306320.jpg,INFECTION,1,cfd2309c24bb86a7b5c81733fc33fe04,4826
INFECTION/306322.jpg,306322.jpg,INFECTION,1,0813078e33641c97c0f83575b131c00c,4827
INFECTION/306353.jpg,306353.jpg,INFECTION,1,8034e2e88315ad2b245582fabb11bad1,4837
INFECTION/306364.jpg,306364.jpg,INFECTION,1,e556ced3e7ef92180cd111661cafa630,4842
INFECTION/306387.jpg,306387.jpg,INFECTION,1,4fe5dbc54ab640d20363d71b7fb1df8f,4849
INFECTION/306402.jpg,306402.jpg,INFECTION,1,9cc3eb61e1cacaaad60450ccda928076,4853
INFECTION/306408.jpg,306408.jpg,INFECTION,1,6fc18dc219e364d4e59ff9187a85a6e6,4854
INFECTION/306421.jpg,306421.jpg,INFECTION,1,11108aeb5fbd49162bc2352dbcd13ef9,4860
INFECTION/306435.jpg,306435.jpg,INFECTION,1,f134e896971a1f92889360def825fdbb,4867
INFECTION/306451.jpg,306451.jpg,INFECTION,1,7ba49920ba2d6935c6495fe86dfb4fb1,4872
INFECTION/306452.jpg,306452.jpg,INFECTION,1,f2bcd7b1f2efd00baf7aa8746047ef31,4873
INFECTION/306456.jpg,306456.jpg,INFECTION,1,a06e270c88df9ee940e657b589a20013,4875
INFECTION/306457.jpg,306457.jpg,INFECTION,1,470c74152c08d210e6cffdf64e98ddb7,4876
INFECTION/306482.jpg,306482.jpg,INFECTION,1,ca4ebb0584efb02f6ee4099fa80ca76d,4888
INFECTION/306492.jpg,306492.jpg,INFECTION,1,f7355e140838d9bb11c7d777c1edb066,4891
INFECTION/306523.jpg,306523.jpg,INFECTION,1,952614c03f2bc063db9be5a022c19a3a,4904
INFECTION/306532.jpg,306532.jpg,INFECTION,1,6e91238931d1e045c0fc3c02613907f8,4909
INFECTION/306538.jpg,306538.jpg,INFECTION,1,3491a848e35a033da97c89128f254dcb,4913
INFECTION/306542.jpg,306542.jpg,INFECTION,1,bb16592b4786973d8cc77f554fb126d9,4915
INFECTION/306570.jpg,306570.jpg,INFECTION,1,e1fa06e63bc4955f61e9b590874eb9ad,4925
INFECTION/306577.jpg,306577.jpg,INFECTION,1,3d02e44ec110d7dfbd4e4c1cdef803fa,4926
INFECTION/306627.jpg,306627.jpg,INFECTION,1,763fcbaf4b190962a1b808730bcdb8f9,4945
INFECTION/306647.jpg,306647.jpg,INFECTION,1,d27b7384014e514cbb24d362593c3d77,4949
INFECTION/306668.jpg,306668.jpg,INFECTION,1,5cd4bc4cb7cb1e4cf08e9daf84b0a75f,4955
INFECTION/306675.jpg,306675.jpg,INFECTION,1,1192cfa802442447eb2da9b48f7ea0a8,4957
INFECTION/306678.jpg,306678.jpg,INFECTION,1,0aadf8211d8d445ab867afb5cfa58fc1,4958
INFECTION/306705.jpg,306705.jpg,INFECTION,1,c3ede37e1b7c6a46fd360c23895022f0,4968
INFECTION/306746.jpg,306746.jpg,INFECTION,1,dd9181c7c83d92c278b4b855fb78f4e4,4984
INFECTION/306774.jpg,306774.jpg,INFECTION,1,dd58d63e8d0963265d612008be0e5322,4995
INFECTION/306786.jpg,306786.jpg,INFECTION,1,05d86a1d83349df87018f982329ce298,5000
INFECTION/306794.jpg,306794.jpg,INFECTION,1,fa13d56ba754b7d377b8007e4eccc766,5004
INFECTION/306796.jpg,306796.jpg,INFECTION,1,93d03c374359e354fa294b8810308dad,5006
INFECTION/306797.jpg,306797.jpg,INFECTION,1,4cb684cac3a272262be6034d85254a7c,5007
INFECTION/306798.jpg,306798.jpg,INFECTION,1,eb91e48e6253fe604a5400696a91de6c,5008
INFECTION/306799.jpg,306799.jpg,INFECTION,1,aaa5c2e5326ae6fdea4b56760d5d1dae,5009
INFECTION/306822.jpg,306822.jpg,INFECTION,1,1a6c35f0a948c09572eec590c9f6e64e,5016
INFECTION/306834.jpg,306834.jpg,INFECTION,1,85191fc73d42930890c416363661094e,5021
INFECTION/306861.jpg,306861.jpg,INFECTION,1,acfe6974099e7d3e6f645b8ec608c6a8,5030
INFECTION/306883.jpg,306883.jpg,INFECTION,1,514c92f78982cc9a2452d896a7bcdda5,5038
INFECTION/306884.jpg,306884.jpg,INFECTION,1,d3c5015a49e1b808b9c53e5adc984fa2,5039
INFECTION/306890.jpg,306890.jpg,INFECTION,1,34bdb88bc639417fb4b76eb2b7585d81,5042
INFECTION/306897.jpg,306897.jpg,INFECTION,1,e6a6b82dacd7ab82c674f0131425faf5,5043
INFECTION/306903.jpg,306903.jpg,INFECTION,1,787380f5094f972c59d2013cf8714b3a,5046
INFECTION/306914.jpg,306914.jpg,INFECTION,1,c040e1f3784fc18d27cb18681696d5b8,5051
INFECTION/306919.jpg,306919.jpg,INFECTION,1,06e6840d338829288b1574f8818d7962,5052
INFECTION/306926.jpg,306926.jpg,INFECTION,1,d95e70bd5731cc081fe43353411d43a0,5058
INFECTION/306942.jpg,306942.jpg,INFECTION,1,a374fd440d89d3b5968d6bb7f5fda267,5065
INFECTION/306944.jpg,306944.jpg,INFECTION,1,81b058951e7112f6050c79bfafbddc34,5066
INFECTION/306948.jpg,306948.jpg,INFECTION,1,356e78cb34356d6862b710ec0f2e6689,5069
INFECTION/306956.jpg,306956.jpg,INFECTION,1,d74a244daf13e9d75405bb35e25f907b,5071
INFECTION/306958.jpg,306958.jpg,INFECTION,1,f57865be5a8404d49dc2ed04fded68aa,5072
INFECTION/306960.jpg,306960.jpg,INFECTION,1,f7c4ff19db5c0df1d4f4c5ceb226b093,5073
INFECTION/306966.jpg,306966.jpg,INFECTION,1,f66533d0110ebb3cd9a5022143253ca8,5076
INFECTION/306971.jpg,306971.jpg,INFECTION,1,d02ed97968e9845f689a0b8612e5b35f,5077
INFECTION/306983.jpg,306983.jpg,INFECTION,1,98307afb5d5fb9c96391af361e95d005,5084
INFECTION/306984.jpg,306984.jpg,INFECTION,1,1774654203cc31c5c543286bd298144d,5085
INFECTION/307002.jpg,307002.jpg,INFECTION,1,d4f82fa7f5b3947640202cc4498f798e,5093
INFECTION/307006.jpg,307006.jpg,INFECTION,1,4e53143f5b914f831a65af5713c41409,5095
INFECTION/307027.jpg,307027.jpg,INFECTION,1,104549b57de064abb94e45b54b619780,5105
ISCHAEMIA/301030.jpg,301030.jpg,ISCHAEMIA,2,2f1d26891f8e61bcd25a438c202212b2,5108
ISCHAEMIA/301168.jpg,301168.jpg,ISCHAEMIA,2,67fd5f45f5c2550ec3071b48707a97e9,5117
ISCHAEMIA/301404.jpg,301404.jpg,ISCHAEMIA,2,56c180c38b4a8a7fd9adffda48d8bcb4,5128
ISCHAEMIA/301553.jpg,301553.jpg,ISCHAEMIA,2,4935dfe3d62dba6909c0067bc5d13803,5136
ISCHAEMIA/301562.jpg,301562.jpg,ISCHAEMIA,2,a4c69798cbefea073154f9a0dff9f91e,5137
ISCHAEMIA/301912.jpg,301912.jpg,ISCHAEMIA,2,1261fa990c915b670e921a8ff535b68d,5146
ISCHAEMIA/302123.jpg,302123.jpg,ISCHAEMIA,2,7067616df7bd55349031d943791e2f8c,5157
ISCHAEMIA/302126.jpg,302126.jpg,ISCHAEMIA,2,9261c548a945956fe4b60bda7e7000e0,5158
ISCHAEMIA/302537.jpg,302537.jpg,ISCHAEMIA,2,d3633d55d09f6aa20df513f4692c7fdb,5171
ISCHAEMIA/302841.jpg,302841.jpg,ISCHAEMIA,2,7d1c4265c54ad386f85b5b00af4ecf2e,5182
ISCHAEMIA/302886.jpg,302886.jpg,ISCHAEMIA,2,e33f2100802de55a1c2701ac80ec238d,5185
ISCHAEMIA/303167.jpg,303167.jpg,ISCHAEMIA,2,c5a24216cd86d2b7b393a998ec650b10,5195
ISCHAEMIA/303212.jpg,303212.jpg,ISCHAEMIA,2,ba711a2ec96d0dff46f43efc1acfe625,5196
ISCHAEMIA/303334.jpg,303334.jpg,ISCHAEMIA,2,988f247fa894deb1f5bda70ec5dabfff,5200
ISCHAEMIA/303431.jpg,303431.jpg,ISCHAEMIA,2,1b4ec5daa9388a624b3952e755616d4d,5205
ISCHAEMIA/303495.jpg,303495.jpg,ISCHAEMIA,2,233153527a3d56a75a04d397a3b0df6e,5207
ISCHAEMIA/303557.jpg,303557.jpg,ISCHAEMIA,2,301f54cbe24d58ce6b4ff7f38a082c91,5209
ISCHAEMIA/303607.jpg,303607.jpg,ISCHAEMIA,2,55335dbbbdf50e0a0b2fb037248e4372,5210
ISCHAEMIA/303715.jpg,303715.jpg,ISCHAEMIA,2,9abf51bdfb49ef35f02e0bcbc5004db6,5214
ISCHAEMIA/303911.jpg,303911.jpg,ISCHAEMIA,2,6157798a96bc99b474ce052a516ffca7,5217
ISCHAEMIA/304239.jpg,304239.jpg,ISCHAEMIA,2,5611d000eb29b65aba8061d8b7eadfa1,5228
ISCHAEMIA/304244.jpg,304244.jpg,ISCHAEMIA,2,8ea9f43b62533f5cd4cfb8bb2983da92,5229
ISCHAEMIA/304325.jpg,304325.jpg,ISCHAEMIA,2,73cdb52b724749756930ed2c62ff82cd,5231
ISCHAEMIA/304402.jpg,304402.jpg,ISCHAEMIA,2,e415d876c97a82e8ec48a39ae5f1f77c,5235
ISCHAEMIA/305017.jpg,305017.jpg,ISCHAEMIA,2,4e727f86ff730d19dc40a7b940dfe599,5255
ISCHAEMIA/305381.jpg,305381.jpg,ISCHAEMIA,2,e17166f19d64cce798c883123f88cdaa,5268
ISCHAEMIA/305459.jpg,305459.jpg,ISCHAEMIA,2,e8b886b55ca4265be07416dcb5bff1fc,5272
ISCHAEMIA/305465.jpg,305465.jpg,ISCHAEMIA,2,3b67a025a097005b83896d60012cea28,5273
ISCHAEMIA/305634.jpg,305634.jpg,ISCHAEMIA,2,0d232df4a2c9b0b827695b55cb5ff85f,5276
ISCHAEMIA/306019.jpg,306019.jpg,ISCHAEMIA,2,3ff3ff21e5472c64bc08a3b4a92daa82,5288
ISCHAEMIA/306083.jpg,306083.jpg,ISCHAEMIA,2,aa56f535853106ee46e10fd429891538,5289
ISCHAEMIA/306092.jpg,306092.jpg,ISCHAEMIA,2,ddf60231a4460d29bf2d0e72092169d4,5290
ISCHAEMIA/306135.jpg,306135.jpg,ISCHAEMIA,2,18498c5917ff9952ad19a595dc9aaa02,5292
ISCHAEMIA/306163.jpg,306163.jpg,ISCHAEMIA,2,18116bad56b4b9c8104d5bbe26a93473,5294
ISCHAEMIA/306168.jpg,306168.jpg,ISCHAEMIA,2,7aad28b9d99c9b63bf4d7d75aec944c2,5295
ISCHAEMIA/306195.jpg,306195.jpg,ISCHAEMIA,2,23a08d6e8cebc4f6347dc54983e0b7f3,5297
ISCHAEMIA/306199.jpg,306199.jpg,ISCHAEMIA,2,445a5fe1090d52eb4b4abfa68215ddee,5298
ISCHAEMIA/306201.jpg,306201.jpg,ISCHAEMIA,2,0ef68150cd93a7f791fc5c4d8dbcf1b8,5299
ISCHAEMIA/306211.jpg,306211.jpg,ISCHAEMIA,2,0c61131264df139ead7b0d1a946d67a1,5300
ISCHAEMIA/306242.jpg,306242.jpg,ISCHAEMIA,2,dbaf82b7bb70fefde54d27a1e5a0c20e,5302
ISCHAEMIA/306290.jpg,306290.jpg,ISCHAEMIA,2,81e901eadc423068da6c7acbeba36fbe,5306
ISCHAEMIA/306314.jpg,306314.jpg,ISCHAEMIA,2,f376f82162cc17ea30c506df01c64fd7,5307
ISCHAEMIA/306404.jpg,306404.jpg,ISCHAEMIA,2,fbe764013da5ab1cb79b182633a93458,5311
ISCHAEMIA/306488.jpg,306488.jpg,ISCHAEMIA,2,832a4b32429c356f0695719f506e25b1,5314
ISCHAEMIA/306684.jpg,306684.jpg,ISCHAEMIA,2,6ae8fe7a3d97ffc2d5e49d89434efe1d,5324
ISCHAEMIA/306726.jpg,306726.jpg,ISCHAEMIA,2,dc2f12dc5fa774e911c0702eb7a78851,5326
BOTH/301038.jpg,301038.jpg,BOTH,3,c7e2c9ae4882eeea7458bdfc193613ad,5337
BOTH/301076.jpg,301076.jpg,BOTH,3,28946e172c83e1598de470661cad7bc7,5342
BOTH/301110.jpg,301110.jpg,BOTH,3,4c9724168b29f7c23d6cd005f0aa398b,5345
BOTH/301167.jpg,301167.jpg,BOTH,3,8a4d640114dc4914c67fcf8eacd05682,5349
BOTH/301193.jpg,301193.jpg,BOTH,3,5ca70fc4a0d459555ee0af51a259d7cb,5350
BOTH/301211.jpg,301211.jpg,BOTH,3,70f9cab34a6748233bf68005d34ea6f7,5352
BOTH/301223.jpg,301223.jpg,BOTH,3,d3799cce7d42e411c5a48256b5c5f31d,5354
BOTH/301372.jpg,301372.jpg,BOTH,3,d70faef01e73a28760da24b8a19edea6,5360
BOTH/301378.jpg,301378.jpg,BOTH,3,afa969132d8bb232d8fc590b8cac6dd3,5363
BOTH/301422.jpg,301422.jpg,BOTH,3,021a2417d6ee92c752df718ca9ca79db,5367
BOTH/301496.jpg,301496.jpg,BOTH,3,88bdb551b262445cb9d85081484c5602,5378
BOTH/301513.jpg,301513.jpg,BOTH,3,14ed84cda58bb9d04ae3afe2951c8b8e,5380
BOTH/301530.jpg,301530.jpg,BOTH,3,02c2d007bbc3ebac05934d2224b13572,5383
BOTH/301558.jpg,301558.jpg,BOTH,3,294650529248c5ad7f3d26f5c67d92a1,5387
BOTH/301648.jpg,301648.jpg,BOTH,3,1ba3f82ed795ce5a6fe2fc5822bc3a96,5396
BOTH/301653.jpg,301653.jpg,BOTH,3,190c9e05a9105bab8e3fa3f09606755f,5397
BOTH/301763.jpg,301763.jpg,BOTH,3,4a0358bfde734f4cb8272a0ad74f3934,5404
BOTH/301773.jpg,301773.jpg,BOTH,3,0b8c33375d7b8f175e781681e4262664,5405
BOTH/301810.jpg,301810.jpg,BOTH,3,d7e9705347c645fedc80a9450ae07c65,5409
BOTH/301891.jpg,301891.jpg,BOTH,3,b4b4e9b90aaf3fe9fe171b42e65c727f,5421
BOTH/301907.jpg,301907.jpg,BOTH,3,76fdf296de14ecd46d00cf9e24c4db89,5423
BOTH/302037.jpg,302037.jpg,BOTH,3,67d0bfde645db6a94562183abbbbac7e,5439
BOTH/302039.jpg,302039.jpg,BOTH,3,d4e3fa066d077d5ff5113065d71b46de,5440
BOTH/302052.jpg,302052.jpg,BOTH,3,74f2070c72d28e22406304eb6a82e2a5,5441
BOTH/302104.jpg,302104.jpg,BOTH,3,a291f2caa27a09f8c83367ccf814d555,5449
BOTH/302173.jpg,302173.jpg,BOTH,3,eb2a0e6051251f63828cf323fe53dfd3,5457
BOTH/302280.jpg,302280.jpg,BOTH,3,f46c9d43da66e997213eb21f6fe27c47,5462
BOTH/302285.jpg,302285.jpg,BOTH,3,3c37c069452ca0ab5df6988468e88499,5463
BOTH/302348.jpg,302348.jpg,BOTH,3,cd387090cb24737644f839184d09c103,5468
BOTH/302369.jpg,302369.jpg,BOTH,3,afaee3563ac63d1538a20517c3b5b984,5471
BOTH/302376.jpg,302376.jpg,BOTH,3,3c7b3537ec81dec1f187f620c0bd6fcc,5474
BOTH/302394.jpg,302394.jpg,BOTH,3,9126efca245172093b734afd988989e5,5477
BOTH/302410.jpg,302410.jpg,BOTH,3,c9594281d132d290fce4bd502e0b13b3,5479
BOTH/302413.jpg,302413.jpg,BOTH,3,57a39a7c00dc9b89700b948b94247583,5480
BOTH/302422.jpg,302422.jpg,BOTH,3,c5be95059a79eb653993a077deb2d666,5481
BOTH/302561.jpg,302561.jpg,BOTH,3,86a93a1de8c2636652fb52c865b1696e,5492
BOTH/302576.jpg,302576.jpg,BOTH,3,fd2addae43ad4a4b5a7d505e3575c139,5496
BOTH/302615.jpg,302615.jpg,BOTH,3,04158644fb97a49467efb66651a7e0da,5497
BOTH/302625.jpg,302625.jpg,BOTH,3,3f2b67d3caf72bcf85b92079133b0115,5498
BOTH/302738.jpg,302738.jpg,BOTH,3,cd736ad4584a3719d383b9b59d356d68,5508
BOTH/302863.jpg,302863.jpg,BOTH,3,c6d2645d93c7d2620286dfd79f6f80db,5513
BOTH/302875.jpg,302875.jpg,BOTH,3,d0be07c2467f072e9c740a361b5b0aed,5514
BOTH/302909.jpg,302909.jpg,BOTH,3,5e7ba68c3f40294975db32a0f7bf49d2,5520
BOTH/302929.jpg,302929.jpg,BOTH,3,aef4068a3d2066956adbed424df22730,5522
BOTH/303035.jpg,303035.jpg,BOTH,3,8d505f121575697c4cb0157fc820d61b,5527
BOTH/303074.jpg,303074.jpg,BOTH,3,d95bddf6c00b589b8992855186064448,5528
BOTH/303116.jpg,303116.jpg,BOTH,3,57acaba53e3244bb7b8355fba8810c4f,5532
BOTH/303285.jpg,303285.jpg,BOTH,3,cca63933d67811e5159d2479a67c2a9a,5554
BOTH/303303.jpg,303303.jpg,BOTH,3,8966fb9e6b492a2a0aa2f8c50209f3e8,5557
BOTH/303309.jpg,303309.jpg,BOTH,3,9a1b547a8b6e14086153aa6b4ea6fcf3,5558
BOTH/303336.jpg,303336.jpg,BOTH,3,7f534d7f4f9ebc3e3afe2f99b52f17a3,5561
BOTH/303443.jpg,303443.jpg,BOTH,3,c2e46d141ae9464647533775ae2f32db,5572
BOTH/303467.jpg,303467.jpg,BOTH,3,8bf7c8eb0a979ef75a1bce9ee654a007,5574
BOTH/303501.jpg,303501.jpg,BOTH,3,d90955f67f2c35c6b077afa1ac0e0ee7,5578
BOTH/303597.jpg,303597.jpg,BOTH,3,9c9188303da95efbdb86f5b7abab79da,5586
BOTH/303649.jpg,303649.jpg,BOTH,3,16a8769798fe80269b3cb305cc4a4037,5593
BOTH/303682.jpg,303682.jpg,BOTH,3,1dac59de1604edc8520f25e64ffc8d25,5595
BOTH/303683.jpg,303683.jpg,BOTH,3,8a24ce6eb13e2adbfcc5a464bfcd0c3f,5596
BOTH/303798.jpg,303798.jpg,BOTH,3,5b91c42c503d048783789d7adc0bdee2,5611
BOTH/303824.jpg,303824.jpg,BOTH,3,7bec38bf5fd86a955c4601bf53f82ae5,5613
BOTH/303828.jpg,303828.jpg,BOTH,3,f170be27eb717d020d6ac5f4bdb09a13,5614
BOTH/303850.jpg,303850.jpg,BOTH,3,fa491fb0674668dc9c07b455ce2ef8a1,5616
BOTH/303886.jpg,303886.jpg,BOTH,3,7f1ae85a3ecff82fd7385eeb64c660e1,5618
BOTH/303955.jpg,303955.jpg,BOTH,3,aa51210fbb2fa02b06b403b55d61e5a4,5621
BOTH/303976.jpg,303976.jpg,BOTH,3,fda213b1c22915ad1c5adc87d83cb0a8,5626
BOTH/303996.jpg,303996.jpg,BOTH,3,c5b116959a929fae59f79223eb2b839d,5629
BOTH/304016.jpg,304016.jpg,BOTH,3,47654b940aa5cb775aed7cef58781645,5631
BOTH/304059.jpg,304059.jpg,BOTH,3,31182456f37c788877b10ca05a89bf70,5633
BOTH/304079.jpg,304079.jpg,BOTH,3,d35ccb1543505bfa9cb1c8f106c62a59,5635
BOTH/304096.jpg,304096.jpg,BOTH,3,c181ae90345b8fbb01c7c55d5d0ebd59,5637
BOTH/304127.jpg,304127.jpg,BOTH,3,95da211f0a68aa6884a71fb3a7b6d025,5639
BOTH/304150.jpg,304150.jpg,BOTH,3,6f8b920629631eb94f1dcbabbc248d2d,5644
BOTH/304156.jpg,304156.jpg,BOTH,3,a6de06f8728c3ff0ecda4733f7f54ab7,5649
BOTH/304256.jpg,304256.jpg,BOTH,3,21409bb4378edd103b663243b621e86a,5654
BOTH/304317.jpg,304317.jpg,BOTH,3,320cce81369c1547030d32b65faac989,5660
BOTH/304368.jpg,304368.jpg,BOTH,3,d54f31a5bb7c9faf36d0bc8f0f535742,5664
BOTH/304384.jpg,304384.jpg,BOTH,3,cdf8c94e1e03f1ef4feeb3d990e4bed5,5666
BOTH/304393.jpg,304393.jpg,BOTH,3,04678f2b21053a7a98935e37f5812fb7,5668
BOTH/304432.jpg,304432.jpg,BOTH,3,b7ef930132a0c1cf6a9693b4c062250d,5672
BOTH/304440.jpg,304440.jpg,BOTH,3,a970df60678b96c5e9186438ab20b8f2,5674
BOTH/304490.jpg,304490.jpg,BOTH,3,4d2a180e846019eaff333d5ce6483d6e,5681
BOTH/304658.jpg,304658.jpg,BOTH,3,90e84acc79f92e4be85c9cbfcea72a79,5697
BOTH/304698.jpg,304698.jpg,BOTH,3,9d6e5aefd63a5abbbfc68fb918989a53,5700
BOTH/304780.jpg,304780.jpg,BOTH,3,162bb6d52554c3cd4024a823367d7d04,5709
BOTH/304857.jpg,304857.jpg,BOTH,3,1a731a9f39dc457b26b83f84b5f6874f,5714
BOTH/304909.jpg,304909.jpg,BOTH,3,2db66c6e7f39885220fe8e86f2d1c584,5717
BOTH/304928.jpg,304928.jpg,BOTH,3,996d941170440d2cf51bf580caecdba9,5719
BOTH/304932.jpg,304932.jpg,BOTH,3,c7cde68c2ad67c268b48cea309991f8f,5721
BOTH/305059.jpg,305059.jpg,BOTH,3,34cdcbe413f864e8906cf9e8c94b6962,5733
BOTH/305153.jpg,305153.jpg,BOTH,3,949ec027431b0867d992980a5c378b8e,5743
BOTH/305205.jpg,305205.jpg,BOTH,3,46747174807cfb25397a48630774d32a,5748
BOTH/305276.jpg,305276.jpg,BOTH,3,0007d7a78ae6d77fe84b34f94f161aa5,5756
BOTH/305299.jpg,305299.jpg,BOTH,3,6a75dce72e0f47f7c67228b03d8b3238,5760
BOTH/305350.jpg,305350.jpg,BOTH,3,5119ddb2b8cf5c6478bd8f809a9e511a,5764
BOTH/305407.jpg,305407.jpg,BOTH,3,883aeed33b87893eb2d920955aed77c6,5769
BOTH/305438.jpg,305438.jpg,BOTH,3,0a887a6aac27ff10829033bfdd5108c9,5774
BOTH/305587.jpg,305587.jpg,BOTH,3,aa4df2c4e39c6bb03cf8cc368edd2766,5792
BOTH/305621.jpg,305621.jpg,BOTH,3,6b895bc37dbf4871d65218b37d95dd3b,5794
BOTH/305685.jpg,305685.jpg,BOTH,3,ef45568f9f57f212080af8a22a336717,5798
BOTH/305723.jpg,305723.jpg,BOTH,3,023b73d94bc939905f7b94632f2abdb6,5804
BOTH/305742.jpg,305742.jpg,BOTH,3,4ca3a51dae4f230b73be9f8cb18ce497,5807
BOTH/305755.jpg,305755.jpg,BOTH,3,59d4f8bd978c49e2145bd7bd830c1f3f,5809
BOTH/305811.jpg,305811.jpg,BOTH,3,c3a830198bdffb8fcef55713ab545542,5811
BOTH/305868.jpg,305868.jpg,BOTH,3,952aee9658e592c6cb8912d3d40f0d5d,5816
BOTH/305971.jpg,305971.jpg,BOTH,3,97d0c6c943900d9a530f0e5d11ccd5f6,5823
BOTH/305994.jpg,305994.jpg,BOTH,3,4fdbc3f7684c79fa6358af409fc54709,5824
BOTH/306139.jpg,306139.jpg,BOTH,3,1d77cb130110b2b7f3c15dcbca69b344,5842
BOTH/306171.jpg,306171.jpg,BOTH,3,c112639c3eef787bc7b87d559c549832,5849
BOTH/306188.jpg,306188.jpg,BOTH,3,157469690f09c5de790b0765c30495fc,5850
BOTH/306209.jpg,306209.jpg,BOTH,3,78463e117e1afbe3abf30fa114dca6ba,5853
BOTH/306280.jpg,306280.jpg,BOTH,3,9a48905a1c4c912598bde743308e2709,5863
BOTH/306289.jpg,306289.jpg,BOTH,3,0b3c43f18920366847029eb69892b629,5867
BOTH/306291.jpg,306291.jpg,BOTH,3,4b50e7cbb93e980ad6f74994e5c434ab,5868
BOTH/306395.jpg,306395.jpg,BOTH,3,111e346f513a4ce983a3e6e8207803e3,5882
BOTH/306396.jpg,306396.jpg,BOTH,3,974873ef9bd10f77901c2b9fcefa8734,5883
BOTH/306410.jpg,306410.jpg,BOTH,3,265c8ea69e691eb77a03d06658133ad3,5885
BOTH/306481.jpg,306481.jpg,BOTH,3,662b05cd7233d03ece559272bbde2a7a,5888
BOTH/306673.jpg,306673.jpg,BOTH,3,57daf6b487b58d362c278da24cdad6d2,5908
BOTH/306766.jpg,306766.jpg,BOTH,3,5e551dabad54604bb9fc201133bbc48f,5924
BOTH/306825.jpg,306825.jpg,BOTH,3,6eb8666858342ce4ba4be7bcd939ca0d,5931
BOTH/306913.jpg,306913.jpg,BOTH,3,e91c45e0817c0c49c8767878c6f5fc96,5941
BOTH/306954.jpg,306954.jpg,BOTH,3,f8a4ab66cb209c2c89a8c5925c7088cc,5944
BOTH/306955.jpg,306955.jpg,BOTH,3,ae2821a4afe7d8dc89fc3cff4fa434dc,5945
BOTH/306975.jpg,306975.jpg,BOTH,3,2aed95fa1445d9bb95c1427985ae98d0,5949


In [ ]:
import pandas as pd
te = pd.read_csv(f"{W}/test.csv"); assert len(te) == 1191, len(te)
missing = [r for r in te.relpath if not os.path.exists(os.path.join(DATA_DIR, r))]
assert not missing, f"{len(missing)} test images not found, e.g. {missing[:3]}"
print("test images found: 1191")


## 3. Benchmark script

In [ ]:
%%writefile /kaggle/working/latency/latency_bench.py
"""Measured complexity and latency of the four variants (DSG, DSG+SE, DSG+HC, MEFF expert),
each as a single network and as a three-network hard-vote ensemble.

Network only : random input, forward passes only (batch 1).
End to end   : one test image -> read JPEG -> (42-D descriptor on the CPU, if the variant uses it) -> resize and
               normalise -> forward pass(es) -> softmax -> hard vote -> class index on the CPU (batch 1).
Weights are random (no checkpoints needed): the compute and therefore the latency do not depend on the weight values.
Usage: python latency_bench.py --code_dir <dir with meff_v3_1.py> --data_dir <DFUC2021_5955> --test_csv split/test.csv --out_dir out
"""
import argparse, json, os, platform, sys, time
import numpy as np, pandas as pd

ap = argparse.ArgumentParser()
for k in ("--code_dir", "--data_dir", "--test_csv", "--out_dir"): ap.add_argument(k, required=True)
ap.add_argument("--n_images", type=int, default=200, help="test images timed end to end")
ap.add_argument("--n_warmup", type=int, default=20)
ap.add_argument("--n_runs", type=int, default=100, help="timed forward passes, network only")
ap.add_argument("--device", default="cuda")
a = ap.parse_args()
sys.path.insert(0, a.code_dir)
import torch
import torchvision
import torchvision.transforms as T
from PIL import Image
from torch.utils.flop_counter import FlopCounterMode
import meff_v3_1 as mv

dev = torch.device("cuda:0" if a.device == "cuda" and torch.cuda.is_available() else "cpu")
os.makedirs(a.out_dir, exist_ok=True)
NETS = {}
for v in mv.VARIANTS:
    NETS[v] = []
    for k in range(3):
        torch.manual_seed(k)
        NETS[v].append(mv.build_model(v, pretrained=False).to(dev).eval())
eval_tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])  # as in meff_v3_1.py

te = pd.read_csv(a.test_csv)
rel = te.sample(n=min(a.n_images, len(te)), random_state=0).relpath.tolist()
paths = [os.path.join(a.data_dir, r) for r in rel]
F = np.stack([mv.handcrafted_42(np.array(Image.open(p).convert("RGB"))) for p in paths])   # only for the z-score constants
mu, sd = F.mean(0), F.std(0); sd[sd < 1e-6] = 1.0                                          # (values do not affect timing)


def sync():
    if dev.type == "cuda":
        torch.cuda.synchronize(dev)


def stats(ts):
    ms = 1000 * np.asarray(ts)
    return float(np.median(ms)), float(np.percentile(ms, 25)), float(np.percentile(ms, 75))


def net_only(v, k, amp):
    models = NETS[v][:k]
    x = torch.randn(1, 3, 224, 224, device=dev)
    h = torch.randn(1, 42, device=dev) if mv.USES_HC[v] else None
    ts = []
    with torch.inference_mode(), torch.autocast(device_type=dev.type, enabled=amp):
        for i in range(a.n_warmup + a.n_runs):
            t0 = time.perf_counter()
            for m in models:
                m(x, h)
            sync()
            if i >= a.n_warmup:
                ts.append(time.perf_counter() - t0)
    return stats(ts)


def end_to_end(v, k, amp):
    models, use_hc = NETS[v][:k], mv.USES_HC[v]
    order = paths[:min(10, len(paths))] + paths          # first images once more as warm-up
    ts, td = [], []
    with torch.inference_mode():
        for i, p in enumerate(order):
            t0 = time.perf_counter()
            img = Image.open(p).convert("RGB")
            hz = None
            if use_hc:
                t1 = time.perf_counter()
                f = (mv.handcrafted_42(np.array(img)) - mu) / sd
                hz = torch.from_numpy(f.astype(np.float32))[None].to(dev)
                d = time.perf_counter() - t1
            x = eval_tf(img)[None].to(dev)
            with torch.autocast(device_type=dev.type, enabled=amp):
                logits = [m(x, hz) for m in models]
            probs = [torch.softmax(l.float(), 1).cpu().numpy() for l in logits]   # .cpu() waits for the GPU
            if k == 1:
                pred = probs[0].argmax(1)
            else:
                pred, _, _ = mv.hard_vote([q.argmax(1) for q in probs], probs)
            t = time.perf_counter() - t0
            if i >= len(order) - len(paths):
                ts.append(t)
                if use_hc:
                    td.append(d)
    return stats(ts), (stats(td) if td else (np.nan, np.nan, np.nan))


rows = []
precisions = [("fp32", False)] + ([("amp_fp16", True)] if dev.type == "cuda" else [])
for v in mv.VARIANTS:
    m0 = NETS[v][0]
    x = torch.randn(1, 3, 224, 224, device=dev); h = torch.randn(1, 42, device=dev) if mv.USES_HC[v] else None
    with torch.no_grad():
        fc = FlopCounterMode(display=False)
        with fc:
            m0(x, h)
    flops = fc.get_total_flops(); prm = sum(p.numel() for p in m0.parameters())
    for k in (1, 3):
        for pname, amp in precisions:
            n_med, n_25, n_75 = net_only(v, k, amp)
            (e_med, e_25, e_75), (d_med, _, _) = end_to_end(v, k, amp)
            rows.append(dict(variant=v, SE=mv.USES_SE[v], handcrafted=mv.USES_HC[v], networks=k, precision=pname,
                             params_M=k * prm / 1e6, gflops=k * flops / 1e9,
                             net_ms_median=n_med, net_ms_p25=n_25, net_ms_p75=n_75,
                             e2e_ms_median=e_med, e2e_ms_p25=e_25, e2e_ms_p75=e_75, descriptor_ms_median=d_med))
            print(f"{v:7s} x{k} {pname:9s} net {n_med:7.2f} ms   end-to-end {e_med:7.2f} ms", flush=True)

df = pd.DataFrame(rows)
df.to_csv(os.path.join(a.out_dir, "latency_all_variants.csv"), index=False)
cpu = platform.processor()
try:
    cpu = next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
except Exception:
    pass
env = dict(python=platform.python_version(), torch=torch.__version__, torchvision=torchvision.__version__,
           cuda=torch.version.cuda, device=str(dev), gpu=torch.cuda.get_device_name(dev) if dev.type == "cuda" else None,
           cpu=cpu, cpu_threads=torch.get_num_threads(), n_images=len(paths), n_warmup=a.n_warmup, n_runs=a.n_runs,
           meff_v3_1_md5=mv.md5_file(os.path.join(a.code_dir, "meff_v3_1.py")), date=time.strftime("%Y-%m-%d %H:%M:%S"))
json.dump(env, open(os.path.join(a.out_dir, "latency_env.json"), "w"), indent=2)
pd.set_option("display.width", 220)
print(df.round(2).to_string(index=False))
print(json.dumps(env, indent=2))


## 4. Run

In [ ]:
!python /kaggle/working/latency/latency_bench.py --code_dir /kaggle/working/latency --data_dir "{DATA_DIR}" --test_csv /kaggle/working/latency/test.csv --out_dir /kaggle/working/latency/out --n_images {N_IMAGES} --n_warmup {N_WARMUP} --n_runs {N_RUNS} --device cuda

In [ ]:
import shutil, json
print(json.load(open(f"{W}/out/latency_env.json"))["gpu"])
shutil.make_archive("/kaggle/working/latency_results", "zip", f"{W}/out")
print("download /kaggle/working/latency_results.zip from the Output tab")
